# Phase 4 · Remaining experiments

Model 2 full fine-tuning, Model 3 AdaLoRA, and Model 4 distillation share the ending-only document policy. The training schedule allows 20 epochs and starts counting three epochs of patience after epoch 10. The session limit saves a resumable checkpoint.

## Required files and environment

Attach `phase4_remaining_kaggle_assets.zip` or its extracted contents.

- The four Phase 1 asset folders with `best.pt`, `tokenizer.json`, and `run_config.json`.
- Official `Phase_4_CJPE/` data or the shared data and token caches. Internet and `HF_TOKEN` are needed to fetch missing data.
- For distillation, a saved Model 1 LoRMA teacher with `best.pt`, `config.json`, `signature.json`, and `progress.json`. Use `TEACHER_DIR` if more than one teacher matches.

Enable a GPU. To resume a run, attach its previous outputs and keep the same settings and GPU count.

## 1. Packages and imports


In [1]:
import subprocess
import sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'peft==0.17.1', 'transformers==4.55.4',
    'huggingface-hub>=0.34,<1', 'scikit-learn>=1.4,<2',
    'ijson>=3.3,<4', 'tqdm>=4.66,<5',
], check=True)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.2/150.2 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 67.1 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'peft==0.17.1', 'transformers==4.55.4', 'huggingface-hub>=0.34,<1', 'scikit-learn>=1.4,<2', 'ijson>=3.3,<4', 'tqdm>=4.66,<5'], returncode=0)

In [2]:
import copy
import gc
import hashlib
import heapq
import itertools
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import time
import zipfile
from collections import Counter, defaultdict
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import HTML, display
import base64
import html
from sklearn.metrics import f1_score, precision_score, recall_score
from torch.utils.checkpoint import checkpoint
from torch.utils.data import DataLoader, Dataset

PAD, UNK, BOS, EOS = range(4)

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary.replace(path)

def digest_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def normalize_id(value):
    value = str(value)
    return value[:-4] if value.endswith(".txt") else value


import contextlib
import importlib.metadata
import ijson
from tqdm.auto import tqdm
from sklearn.metrics import confusion_matrix, classification_report
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from IPython.display import FileLink

VERSION = 'phase4_cjpe_v1'
TRAINER_VERSION = 'phase4_remaining_min10_v1'
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name}, {p.total_memory / 2**30:.2f} GiB')


PyTorch: 2.11.0+cu128 | CUDA: True
GPU 0: Tesla T4, 14.56 GiB
GPU 1: Tesla T4, 14.56 GiB


## 2. Settings

Each run uses the last 1,024 tokens. Set the epoch budget before training and reuse it when resuming.

In [3]:
INPUT_ROOT = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working/phase4_remaining')
ASSET_DIR = OUTPUT_ROOT / 'assets'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ASSET_DIR.mkdir(parents=True, exist_ok=True)

DATA_ROOT = None
PREPARED_CACHE = None
TEACHER_DIR = None
TEACHER_RUN_NAME = None
MODEL_NAMES = ['model1', 'model2', 'model3', 'model4']
MODEL_DIRS = dict.fromkeys(MODEL_NAMES)
REMAINING_RUN_NAMES = ['model2_full_min10', 'model3_adalora_min10', 'model4_distilled_min10']
REPO = 'Exploration-Lab/CS-781-Capstone'
DATA_REVISION = None
LABEL_FIELD = None

DOCUMENT_POLICY = dict(representation='tail', aggregator='attention',
                       chunk_tokens=1024, overlap=128, max_chunks=6)
DEFAULT = dict(
    seed=42, **DOCUMENT_POLICY, head_dim=128, dropout=0.2,
    epochs=20, min_epochs=10, patience=3,
    effective_batch=8, micro_batch_per_gpu=1, activation_checkpointing=True,
    backbone_lr=2e-5, adapter_lr=1e-4, head_lr=2e-4,
    weight_decay=0.01, warmup_fraction=0.06, min_lr_ratio=0.1,
    clip=1.0, label_smoothing=0.0, class_weighting=False,
    rank=8, initial_rank=12, alpha=16, orth_reg_weight=0.5,
    adalora_schedule_epochs=10,
    model2_pool='encoder_query', save_every=100, log_every=20,
    max_hours=10.5, capacity_headroom_gib=1.5,
    amp_initial_scale=1024.0, max_consecutive_amp_skips=12,
    kd_alpha=0.5, kd_temperature=2.0, student_keep=None,
    trainer_version=TRAINER_VERSION,
)
TEACHER_FOLDERS = {}
PREPARED_CACHES = []
SEARCH_ROOTS = [INPUT_ROOT]
print('Output:', OUTPUT_ROOT)
print('Epoch policy:', {key: DEFAULT[key] for key in ('epochs', 'min_epochs', 'patience')})


Output: /kaggle/working/phase4_remaining
Epoch policy: {'epochs': 20, 'min_epochs': 10, 'patience': 3}


## 3. Locate the attached files


In [4]:
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
    del token
except Exception:
    pass


def classify_model_config(config):
    if 'encoder_layers' in config and 'decoder_layers' in config:
        return 'model2'
    if 'kernel_size' in config and 'dilation_cycle' in config:
        return 'model4'
    if 'window' in config and 'global_every' in config:
        return 'model3'
    if 'layers' in config and 'heads' in config and 'context' in config:
        return 'model1'
    return None

def unpack_attached_backups():
    """Import recognized Phase 4 ZIPs if Kaggle has left them compressed."""
    import_root = OUTPUT_ROOT / 'attached_backups'
    restored = []
    for archive_path in sorted(INPUT_ROOT.rglob('*.zip')):
        prefixes = ('phase4_shared_', 'model1_lorma_', 'phase4_remaining_kaggle_assets') + tuple(name + '_' for name in REMAINING_RUN_NAMES)
        if not archive_path.name.startswith(prefixes):
            continue
        with zipfile.ZipFile(archive_path) as archive:
            selected = []
            for item in archive.infolist():
                parts = Path(item.filename).parts
                if item.is_dir() or not parts or '..' in parts or Path(item.filename).is_absolute():
                    continue
                shared = len(parts) > 2 and parts[:2] in (
                    ('phase4_cjpe', 'assets'), ('phase4_cjpe', 'token_cache'),
                    ('phase4_cjpe', 'teacher_cache'))
                phase1 = (len(parts) == 3 and parts[0] == 'phase3_model_assets'
                          and parts[-1] in ('best.pt', 'tokenizer.json', 'run_config.json'))
                resumed_run = (len(parts) >= 4 and parts[:2] == ('phase4_cjpe', 'runs')
                               and parts[2] in REMAINING_RUN_NAMES)
                teacher = (len(parts) == 4 and parts[:2] == ('phase4_cjpe', 'runs')
                           and item.filename.rsplit('/', 1)[-1] in (
                               'best.pt', 'config.json', 'signature.json', 'progress.json'))
                if shared or phase1 or teacher or resumed_run:
                    selected.append(item)
            if not selected:
                continue
            destination = import_root / archive_path.stem
            marker = destination / 'imported.json'
            identity = dict(source=str(archive_path), bytes=archive_path.stat().st_size,
                            modified=archive_path.stat().st_mtime_ns)
            if marker.is_file() and read_json(marker) == identity:
                restored.append(destination)
                continue
            needed = sum(item.file_size for item in selected)
            if shutil.disk_usage(OUTPUT_ROOT).free < needed + 256 * 2**20:
                raise OSError(f'Not enough space to unpack {archive_path.name}. Attach it as an extracted dataset.')
            for item in tqdm(selected, desc=f'Import {archive_path.stem}'):
                target = destination / item.filename
                if not target.resolve().is_relative_to(destination.resolve()):
                    raise ValueError('Archive path escapes its destination')
                target.parent.mkdir(parents=True, exist_ok=True)
                temporary = target.with_suffix(target.suffix + '.importing')
                with archive.open(item) as incoming, temporary.open('wb') as outgoing:
                    shutil.copyfileobj(incoming, outgoing, length=8 * 2**20)
                temporary.replace(target)
            write_json(marker, identity)
            restored.append(destination)
    return restored


def valid_cjpe_folder(root):
    return all((root / name).is_file() for name in (
        'Train/train.json', 'Train/dev.json', 'Test/test.json'))


def resolve_remaining_inputs():
    global PREPARED_CACHES, SEARCH_ROOTS
    restored = unpack_attached_backups()
    SEARCH_ROOTS = [INPUT_ROOT] + restored
    candidates = defaultdict(list)
    for root in SEARCH_ROOTS:
        for path in root.rglob('run_config.json'):
            if all((path.parent / name).is_file() for name in ('best.pt', 'tokenizer.json')):
                kind = classify_model_config(read_json(path).get('model', {}))
                if kind:
                    candidates[kind].append(path.parent)
    for name in MODEL_NAMES:
        if MODEL_DIRS[name] is None:
            options = sorted(set(candidates[name]))
            if len(options) != 1:
                raise ValueError(f'Set MODEL_DIRS[{name!r}] explicitly. Found: {options}')
            MODEL_DIRS[name] = options[0]
        MODEL_DIRS[name] = Path(MODEL_DIRS[name])
        for filename in ('best.pt', 'tokenizer.json', 'run_config.json'):
            if not (MODEL_DIRS[name] / filename).is_file():
                raise FileNotFoundError(MODEL_DIRS[name] / filename)

    destination = ASSET_DIR / 'Phase_4_CJPE'
    if DATA_ROOT is not None:
        source = Path(DATA_ROOT)
    elif valid_cjpe_folder(destination):
        source = destination
    else:
        options = sorted(set(path.parent.parent for root in SEARCH_ROOTS
                             for path in root.rglob('train.json')
                             if path.parent.name == 'Train' and valid_cjpe_folder(path.parent.parent)))
        if options:
            hashes = {tuple(digest_file(folder / n) for n in (
                'Train/train.json', 'Train/dev.json', 'Test/test.json')) for folder in options}
            if len(hashes) != 1:
                raise ValueError(f'Multiple different CJPE copies. Set DATA_ROOT explicitly: {options}')
            source = options[0]
        else:
            from huggingface_hub import snapshot_download
            try:
                snapshot = Path(snapshot_download(repo_id=REPO, repo_type='dataset',
                    revision=DATA_REVISION, allow_patterns=['Phase_4_CJPE/**']))
            except Exception as error:
                raise RuntimeError('Attach the shared Phase 4 data backup, or enable Internet with an authorized HF_TOKEN.') from error
            source = snapshot / 'Phase_4_CJPE'
    if not valid_cjpe_folder(source):
        raise FileNotFoundError(f'Incomplete CJPE data: {source}')
    for name in ('Train/train.json', 'Train/dev.json', 'Test/test.json'):
        target = destination / name
        target.parent.mkdir(parents=True, exist_ok=True)
        if source.resolve() != destination.resolve():
            if not target.is_file() or digest_file(target) != digest_file(source / name):
                shutil.copy2(source / name, target)

    roots = [Path(PREPARED_CACHE)] if PREPARED_CACHE is not None else []
    for root in SEARCH_ROOTS:
        roots.extend(path for path in root.rglob('token_cache') if path.is_dir())
    PREPARED_CACHES = sorted(set(roots))
    print('Data:', destination)
    for name, folder in MODEL_DIRS.items():
        print(name, folder)
    print('Attached token-cache folders:', PREPARED_CACHES)
    return destination


def restore_saved_remaining_runs():
    for name in REMAINING_RUN_NAMES:
        options = []
        for root in SEARCH_ROOTS:
            for path in root.rglob('config.json'):
                if not (path.parent / 'latest.pt').is_file():
                    continue
                try:
                    spec = read_json(path)
                except (ValueError, OSError):
                    continue
                if spec.get('name') == name:
                    options.append(path.parent)
        if not options:
            continue
        options = sorted(set(options))
        if len(options) > 1:
            hashes = {digest_file(folder / 'latest.pt') for folder in options}
            if len(hashes) != 1:
                raise ValueError(f'Multiple resume checkpoints for {name}. Attach one version: {options}')
        source = options[0]
        destination = OUTPUT_ROOT / 'runs' / name
        files = [path for path in source.rglob('*') if path.is_file() and path.suffix != '.tmp']
        missing = [path for path in files if not (destination / path.relative_to(source)).exists()]
        needed = sum(path.stat().st_size for path in missing)
        if shutil.disk_usage(OUTPUT_ROOT).free < needed + 256 * 2**20:
            raise OSError(f'Not enough working space to restore {name}.')
        for path in tqdm(missing, desc=f'Restore {name}'):
            target = destination / path.relative_to(source)
            target.parent.mkdir(parents=True, exist_ok=True)
            temporary = target.with_suffix(target.suffix + '.restoring')
            shutil.copy2(path, temporary)
            temporary.replace(target)
        print('Writable resume folder:', destination)


def discover_teacher_folders():
    folders = defaultdict(list)
    for root in SEARCH_ROOTS:
        for path in root.rglob('config.json'):
            folder = path.parent
            if not all((folder / n).is_file() for n in ('best.pt', 'signature.json', 'progress.json')):
                continue
            try:
                spec = read_json(path)
            except (ValueError, OSError):
                continue
            if spec.get('model') in MODEL_NAMES and spec.get('method') in ('full', 'lorma', 'adalora'):
                folders[spec['name']].append(folder)
    if TEACHER_DIR is not None:
        folder = Path(TEACHER_DIR)
        spec = read_json(folder / 'config.json')
        folders[spec['name']] = [folder]
    resolved = {}
    for name, options in folders.items():
        options = sorted(set(options))
        if len(options) > 1:
            hashes = {digest_file(folder / 'best.pt') for folder in options}
            if len(hashes) != 1:
                raise ValueError(f'Different teacher checkpoints share the name {name}. Set TEACHER_DIR explicitly: {options}')
        resolved[name] = options[0]
    for name, folder in resolved.items():
        progress = read_json(folder / 'progress.json')
        print('Attached teacher:', name, '| best dev F1:', progress['best_macro_f1'])
    return resolved


In [5]:
DATA_ROOT = resolve_remaining_inputs()
restore_saved_remaining_runs()
TEACHER_FOLDERS = discover_teacher_folders()
write_json(ASSET_DIR / 'environment.json', dict(
    version=VERSION, trainer=TRAINER_VERSION, torch=torch.__version__,
    packages={p: importlib.metadata.version(p) for p in
              ['peft', 'transformers', 'huggingface-hub', 'scikit-learn', 'ijson']},
    data_hashes={name: digest_file(DATA_ROOT / name) for name in
                 ['Train/train.json', 'Train/dev.json', 'Test/test.json']},
))


Data: /kaggle/working/phase4_remaining/assets/Phase_4_CJPE
model1 /kaggle/input/datasets/acadsiitk/phase4-assets/phase3_model_assets/model1_decoder
model2 /kaggle/input/datasets/acadsiitk/phase4-assets/phase3_model_assets/model2_encoder_decoder
model3 /kaggle/input/datasets/acadsiitk/phase4-assets/phase3_model_assets/model3_local_global
model4 /kaggle/input/datasets/acadsiitk/phase4-assets/phase3_model_assets/model4_gated_cnn
Attached token-cache folders: [PosixPath('/kaggle/input/datasets/acadsiitk/phase4-assets/phase4_cjpe/token_cache')]
Attached teacher: model1_lorma_to20 | best dev F1: 0.7076123004247933


## 4. Audit the corpus

Identical repeated IDs are collapsed. Conflicting repeated IDs stop for inspection. Training copies of development/test text and duplicate training text are excluded using the same policy as the previous notebook. Development and test IDs remain unchanged.


In [6]:
def iter_records(path):
    with open(path, 'rb') as f:
        first = f.read(4096).lstrip(b'\xef\xbb\xbf \r\n\t')[:1]
    with open(path, 'rb') as f:
        if first == b'[':
            yield from ijson.items(f, 'item')
        elif first == b'{':
            for key, value in ijson.kvitems(f, ''):
                if not isinstance(value, dict):
                    raise ValueError(f'{path}: expected records keyed by case ID, found {key!r}')
                record = dict(value)
                record.setdefault('id', key)
                yield record
        else:
            raise ValueError(f'{path}: expected a JSON list or keyed dictionary')

def clean_record(record, labelled):
    if 'id' not in record or 'text' not in record:
        raise ValueError(f'Record needs id/text. Available keys: {list(record)}')
    identifier = record['id']
    if not isinstance(identifier, str) or not identifier.strip():
        raise ValueError(f'Expected a nonempty string case ID, got {identifier!r}')
    text = record['text']
    if not isinstance(text, str) or not text.strip():
        raise ValueError(f'Case {identifier}: text must be a nonempty string')
    label = None
    if labelled:
        keys = [LABEL_FIELD] if LABEL_FIELD else [k for k in ('label','gold','outcome','judgment') if k in record]
        if len(keys) != 1 or keys[0] not in record:
            raise ValueError(f'Set LABEL_FIELD. Case {identifier} has keys {list(record)}')
        raw = record[keys[0]]
        if isinstance(raw, bool) or raw not in (0, 1, '0', '1'):
            raise ValueError(f'Case {identifier}: unsupported binary label {raw!r}')
        label = int(raw)
    fingerprint = hashlib.sha256(' '.join(text.split()).encode('utf-8')).hexdigest()
    return identifier, text, label, fingerprint

def audit_corpus():
    index, report, exclusions = {}, {}, []
    blocked = set()
    for split in ('dev','test','train'):
        path = DATA_ROOT / ('Test/test.json' if split == 'test' else f'Train/{split}.json')
        rows, ids, seen_text, labels, lengths = [], {}, {}, Counter(), []
        duplicate_ids = 0
        for record in tqdm(iter_records(path), desc=f'Audit {split}'):
            identifier, text, label, text_hash = clean_record(record, split != 'test')
            identity = (hashlib.sha256(text.encode('utf-8')).hexdigest(), label)
            if identifier in ids:
                if ids[identifier] != identity:
                    issue = dict(split=split, id=identifier,
                        first_text_sha256=ids[identifier][0], first_label=ids[identifier][1],
                        repeated_text_sha256=identity[0], repeated_label=label)
                    write_json(ASSET_DIR / 'conflicting_id.json', issue)
                    raise ValueError(f'Repeated ID has different text or labels: {issue}. Inspect the source records before choosing a policy.')
                duplicate_ids += 1
                exclusions.append(dict(split=split,id=identifier,reason='identical_duplicate_id'))
                continue
            ids[identifier] = identity
            if split == 'train':
                if text_hash in seen_text and seen_text[text_hash] != label:
                    raise ValueError(f'Conflicting training labels for identical text: {identifier}')
                repeated = text_hash in seen_text
                seen_text[text_hash] = label
                if text_hash in blocked or repeated:
                    exclusions.append({'split':split,'id': identifier, 'reason': 'heldout_text' if text_hash in blocked else 'duplicate_train_text'})
                    continue
            else:
                blocked.add(text_hash)
            rows.append(dict(id=identifier, label=label, text_sha256=text_hash))
            labels[str(label)] += 1
            lengths.append(len(text.split()))
        if not rows: raise ValueError(f'Empty {split} split')
        index[split] = rows
        report[split] = dict(examples=len(rows), labels=dict(labels), identical_duplicate_ids=duplicate_ids,
            word_percentiles=dict(zip(['50','90','95','99','100'], np.percentile(lengths,[50,90,95,99,100]).tolist())))
    if set(r['label'] for r in index['train']) != {0,1}:
        raise ValueError('Training must contain both classes')
    report['excluded_training_records'] = sum(r['split']=='train' for r in exclusions)
    write_json(ASSET_DIR / 'corpus_audit.json', report)
    write_json(ASSET_DIR / 'excluded_train.json', [r for r in exclusions if r['split']=='train'])
    write_json(ASSET_DIR / 'duplicate_and_exclusion_audit.json', exclusions)
    write_json(ASSET_DIR / 'case_index.json', index)
    write_json(ASSET_DIR / 'dev_truth.json', [dict(id=r['id'],gold=r['label']) for r in index['dev']])
    print(json.dumps(report, indent=2))
    return index


In [7]:
CASE_INDEX = audit_corpus()


Audit dev: 0it [00:00, ?it/s]

Audit test: 0it [00:00, ?it/s]

Audit train: 0it [00:00, ?it/s]

{
  "dev": {
    "examples": 2511,
    "labels": {
      "0": 1252,
      "1": 1259
    },
    "identical_duplicate_ids": 994,
    "word_percentiles": {
      "50": 2909.0,
      "90": 7210.0,
      "95": 9634.5,
      "99": 18266.100000000002,
      "100": 42198.0
    }
  },
  "test": {
    "examples": 1517,
    "labels": {
      "None": 1517
    },
    "identical_duplicate_ids": 0,
    "word_percentiles": {
      "50": 2940.0,
      "90": 7302.400000000001,
      "95": 10023.400000000001,
      "99": 18235.599999999988,
      "100": 40487.0
    }
  },
  "train": {
    "examples": 31653,
    "labels": {
      "0": 18539,
      "1": 13114
    },
    "identical_duplicate_ids": 5082,
    "word_percentiles": {
      "50": 2030.0,
      "90": 6430.799999999999,
      "95": 8986.399999999998,
      "99": 19272.759999999995,
      "100": 412606.0
    }
  },
  "excluded_training_records": 5734
}


## 5. Original tokenizer and Phase 1 models

These definitions match your four Phase 1 checkpoints. The tokenizer is loaded from your saved file; it is not retrained.


In [8]:
def split_bpe_pieces(text, max_chars=256):
    """Keep every character; separate words, punctuation and whitespace.

    An optional leading ASCII space can join a word or punctuation piece.
    Long pieces are split into bounded chunks to limit per-piece work.
    No normalization, strip(), split(), external tokenizer or regex package.
    """
    for match in re.finditer(' ?\\w+| ?[^\\w\\s]+|\\s+', text):
        piece = match.group(0)
        for start in range(0, len(piece), max_chars):
            yield piece[start:start + max_chars].encode('utf-8')


def apply_bpe_merges(piece, merge_ranks):
    """Apply learned merges by rank, breaking occurrence ties left-to-right.

    Token IDs 0..3 are reserved. Byte b always has ID b+4. A linked list
    and a heap avoid repeatedly scanning every adjacent pair in a piece.
    """
    ids = [byte + 4 for byte in piece]
    size = len(ids)
    if size < 2 or not merge_ranks:
        return tuple(ids)
    previous = [i - 1 for i in range(size)]
    following = [i + 1 for i in range(size)]
    following[-1] = -1
    active = [True] * size
    queue = []

    def offer(left):
        if left < 0 or not active[left]:
            return
        right = following[left]
        if right < 0:
            return
        pair = (ids[left], ids[right])
        rank = merge_ranks.get(pair)
        if rank is not None:
            heapq.heappush(queue, (rank, left, right, pair[0], pair[1]))
    for left in range(size - 1):
        offer(left)
    while queue:
        rank, left, right, a, b = heapq.heappop(queue)
        if not active[left] or not active[right] or following[left] != right:
            continue
        if ids[left] != a or ids[right] != b:
            continue
        ids[left] = 260 + rank
        active[right] = False
        following[left] = following[right]
        if following[right] >= 0:
            previous[following[right]] = left
        offer(previous[left])
        offer(left)
    result = []
    position = 0
    while position >= 0:
        result.append(ids[position])
        position = following[position]
    return tuple(result)


class ScratchByteBPE:
    """Our byte BPE vocabulary, encoder, decoder and JSON serialization.

    All base bytes and EVERY intermediate merge token are retained.
    Integer IDs identify symbols; concatenated decimal strings are never used.
    Literal '<s>' etc. in ordinary input are encoded as text, not control IDs.
    """
    FORMAT = 'handwritten-byte-bpe-v1'
    PATTERN = ' ?\\w+| ?[^\\w\\s]+|\\s+'

    def __init__(self, merges=None, max_piece_chars=256):
        self.specials = ['<pad>', '<unk>', '<s>', '</s>']
        self.max_piece_chars = max_piece_chars
        if not isinstance(max_piece_chars, int) or max_piece_chars < 1:
            raise ValueError('max_piece_chars must be a positive integer')
        self.merges = []
        self.merge_ranks = {}
        self.token_bytes = [None] * 4 + [bytes([b]) for b in range(256)]
        self._cache = {}
        for pair in merges or []:
            self.add_merge(tuple(pair))

    def add_merge(self, pair):
        if len(pair) != 2 or pair in self.merge_ranks:
            raise ValueError('Invalid or repeated merge pair')
        if any((type(token) is not int or not 4 <= token < len(self.token_bytes) for token in pair)):
            raise ValueError('A merge must reference earlier non-special token IDs')
        new_id = len(self.token_bytes)
        self.merge_ranks[pair] = len(self.merges)
        self.merges.append(pair)
        self.token_bytes.append(self.token_bytes[pair[0]] + self.token_bytes[pair[1]])
        self._cache.clear()
        return new_id

    def get_vocab_size(self):
        return len(self.token_bytes)

    def token_to_id(self, special):
        return self.specials.index(special) if special in self.specials else None

    def encode(self, text):
        if not isinstance(text, str):
            raise TypeError('encode expects a string')
        output = []
        for piece in split_bpe_pieces(text, self.max_piece_chars):
            encoded = self._cache.get(piece)
            if encoded is None:
                encoded = apply_bpe_merges(piece, self.merge_ranks)
                if len(self._cache) >= 65536:
                    self._cache.clear()
                self._cache[piece] = encoded
            output.extend(encoded)
        return SimpleNamespace(ids=output)

    def decode(self, ids, skip_special_tokens=True):
        chunks = []
        for value in ids:
            token = int(value)
            if not 0 <= token < len(self.token_bytes):
                raise ValueError(f'Unknown token ID: {token}')
            if token < 4:
                if not skip_special_tokens:
                    chunks.append(self.specials[token].encode('utf-8'))
            else:
                chunks.append(self.token_bytes[token])
        return b''.join(chunks).decode('utf-8', errors='replace')

    def save(self, path, training=None):
        payload = {'format': self.FORMAT, 'special_tokens': self.specials, 'byte_offset': 4, 'pattern': self.PATTERN, 'max_piece_chars': self.max_piece_chars, 'merges': self.merges, 'vocab_hex': [None if value is None else value.hex() for value in self.token_bytes]}
        if training is not None:
            payload['training'] = training
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_suffix(path.suffix + '.tmp')
        temporary.write_text(json.dumps(payload, ensure_ascii=False), encoding='utf-8')
        temporary.replace(path)

    @classmethod
    def from_file(cls, path):
        payload = json.loads(Path(path).read_text(encoding='utf-8'))
        if payload.get('format') != cls.FORMAT:
            raise ValueError('This is not our handwritten BPE format. Use a fresh data directory and retrain the tokenizer/token arrays.')
        if payload.get('special_tokens') != ['<pad>', '<unk>', '<s>', '</s>']:
            raise ValueError('Special-token IDs do not match')
        if payload.get('byte_offset') != 4 or payload.get('pattern') != cls.PATTERN:
            raise ValueError('Byte mapping or splitting rules do not match')
        tokenizer = cls(payload['merges'], payload['max_piece_chars'])
        expected = [None if value is None else value.hex() for value in tokenizer.token_bytes]
        if payload.get('vocab_hex') != expected:
            raise ValueError('Vocabulary is inconsistent with the merge history')
        return tokenizer


In [9]:
class DecoderAttention(nn.Module):

    def __init__(self, width, heads, dropout):
        super().__init__()
        assert width % heads == 0
        self.heads, self.dim, self.dropout = (heads, width // heads, dropout)
        self.qkv = nn.Linear(width, 3 * width)
        self.proj = nn.Linear(width, width)

    def forward(self, x, past=None, cache=False):
        b, t, c = x.shape
        q, k, v = self.qkv(x).view(b, t, 3, self.heads, self.dim).unbind(2)
        q, k, v = (a.transpose(1, 2) for a in (q, k, v))
        if past is not None:
            if t != 1:
                raise ValueError('Cached decoding accepts one new token at a time')
            k, v = (torch.cat((past[0], k), 2), torch.cat((past[1], v), 2))
        y = F.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=past is None)
        return (self.proj(y.transpose(1, 2).contiguous().view(b, t, c)), (k, v) if cache else None)


class Block(nn.Module):

    def __init__(self, width, heads, dropout):
        super().__init__()
        self.ln1, self.ln2 = (nn.LayerNorm(width), nn.LayerNorm(width))
        self.attn = DecoderAttention(width, heads, dropout)
        self.mlp = nn.Sequential(nn.Linear(width, 4 * width), nn.GELU(), nn.Linear(4 * width, width))
        self.drop = nn.Dropout(dropout)

    def forward(self, x, past=None, cache=False):
        a, kv = self.attn(self.ln1(x), past, cache)
        x = x + self.drop(a)
        return (x + self.drop(self.mlp(self.ln2(x))), kv)


class DecoderLM(nn.Module):

    def __init__(self, vocab_size=32768, width=704, layers=16, heads=11, context=2048, dropout=0.0):
        super().__init__()
        self.config = dict(vocab_size=vocab_size, width=width, layers=layers, heads=heads, context=context, dropout=dropout)
        self.token = nn.Embedding(vocab_size, width)
        self.position = nn.Embedding(context, width)
        self.blocks = nn.ModuleList([Block(width, heads, dropout) for _ in range(layers)])
        self.final_norm = nn.LayerNorm(width)
        self.drop = nn.Dropout(dropout)
        self.gradient_checkpointing = True
        self.apply(self._init)
        for block in self.blocks:
            nn.init.normal_(block.attn.proj.weight, std=0.02 / math.sqrt(2 * layers))
            nn.init.normal_(block.mlp[2].weight, std=0.02 / math.sqrt(2 * layers))

    @staticmethod
    def _init(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, ids, targets=None, past=None, cache=False):
        offset = 0 if past is None else past[0][0].shape[2]
        if offset + ids.shape[1] > self.config['context']:
            raise ValueError('Context window exceeded')
        x = self.drop(self.token(ids) + self.position(torch.arange(offset, offset + ids.shape[1], device=ids.device)))
        kvs = []
        for i, block in enumerate(self.blocks):
            if self.training and self.gradient_checkpointing:
                x = checkpoint(lambda z, b=block: b(z)[0], x, use_reentrant=False)
            else:
                x, kv = block(x, None if past is None else past[i], cache)
                if cache:
                    kvs.append(kv)
        x = self.final_norm(x)
        if targets is not None:
            flat, labels = (x.reshape(-1, x.shape[-1]), targets.reshape(-1))

            def chunk_loss(h, y):
                return F.cross_entropy(F.linear(h, self.token.weight).float(), y, ignore_index=-100, reduction='sum')
            loss = x.new_zeros((), dtype=torch.float32)
            for start in range(0, len(labels), 256):
                h, y = (flat[start:start + 256], labels[start:start + 256])
                loss = loss + (checkpoint(chunk_loss, h, y, use_reentrant=False) if self.training else chunk_loss(h, y))
            return loss
        return (F.linear(x[:, -1], self.token.weight), kvs)


In [10]:
class EncDecAttention(nn.Module):
    def __init__(self,width,heads,dropout=0.0):
        super().__init__(); assert width%heads==0
        self.heads=heads; self.dim=width//heads; self.dropout=dropout
        self.q=nn.Linear(width,width); self.k=nn.Linear(width,width)
        self.v=nn.Linear(width,width); self.out=nn.Linear(width,width)

    def forward(self,x,memory=None,valid=None,causal=False,past=None,cache=False):
        b,t,w=x.shape
        def split(z): return z.view(b,-1,self.heads,self.dim).transpose(1,2)
        q=split(self.q(x))
        if memory is not None and past is not None:
            k,v=past
        else:
            source=x if memory is None else memory
            k,v=split(self.k(source)),split(self.v(source))
            if past is not None:
                k=torch.cat((past[0],k),dim=2); v=torch.cat((past[1],v),dim=2)
        mask=None
        if valid is not None: mask=valid[:,None,None,:]
        if causal:
            offset=k.shape[2]-t
            allowed=torch.arange(k.shape[2],device=x.device)[None,:] <= (torch.arange(t,device=x.device)+offset)[:,None]
            mask=allowed[None,None,:,:] if mask is None else mask & allowed[None,None,:,:]
        h=F.scaled_dot_product_attention(q,k,v,attn_mask=mask,
            dropout_p=self.dropout if self.training else 0.0)
        h=self.out(h.transpose(1,2).contiguous().view(b,t,w))
        return h, (k,v) if cache else None

class EncoderBlock(nn.Module):
    def __init__(self,w,h,p):
        super().__init__(); self.n1=nn.LayerNorm(w); self.n2=nn.LayerNorm(w)
        self.attn=EncDecAttention(w,h,p); self.drop=nn.Dropout(p)
        self.ff=nn.Sequential(nn.Linear(w,4*w),nn.GELU(),nn.Linear(4*w,w))
    def forward(self,x,valid):
        x=x+self.drop(self.attn(self.n1(x),valid=valid)[0])
        return x+self.drop(self.ff(self.n2(x)))

class DecoderBlock(nn.Module):
    def __init__(self,w,h,p):
        super().__init__(); self.n1=nn.LayerNorm(w); self.n2=nn.LayerNorm(w); self.n3=nn.LayerNorm(w)
        self.self_attn=EncDecAttention(w,h,p); self.cross=EncDecAttention(w,h,p); self.drop=nn.Dropout(p)
        self.ff=nn.Sequential(nn.Linear(w,4*w),nn.GELU(),nn.Linear(4*w,w))
    def forward(self,x,memory,source_valid,target_valid=None,past=None,cache=False):
        self_past,cross_past=(None,None) if past is None else past
        a,skv=self.self_attn(self.n1(x),valid=target_valid,causal=True,past=self_past,cache=cache)
        x=x+self.drop(a)
        a,ckv=self.cross(self.n2(x),memory=memory,valid=source_valid,past=cross_past,cache=cache)
        x=x+self.drop(a); x=x+self.drop(self.ff(self.n3(x)))
        return x, (skv,ckv) if cache else None


class EncoderDecoder(nn.Module):
    def __init__(self,vocab_size=32768,width=704,encoder_layers=6,decoder_layers=8,heads=11,dropout=0.0):
        super().__init__()
        self.config=dict(vocab_size=vocab_size,width=width,encoder_layers=encoder_layers,
                         decoder_layers=decoder_layers,heads=heads,dropout=dropout)
        self.token=nn.Embedding(vocab_size,width)
        self.encoder=nn.ModuleList([EncoderBlock(width,heads,dropout) for _ in range(encoder_layers)])
        self.decoder=nn.ModuleList([DecoderBlock(width,heads,dropout) for _ in range(decoder_layers)])
        self.enc_norm=nn.LayerNorm(width); self.dec_norm=nn.LayerNorm(width)
        self.drop=nn.Dropout(dropout); self.gradient_checkpointing=True
        self.apply(self.initialize)

    @staticmethod
    def initialize(m):
        if isinstance(m,(nn.Linear,nn.Embedding)):
            nn.init.normal_(m.weight,std=0.02)
            if isinstance(m,nn.Linear) and m.bias is not None: nn.init.zeros_(m.bias)

    def embed(self,ids,offset=0):
        w=self.config['width']; positions=torch.arange(offset,offset+ids.shape[1],device=ids.device).float()
        frequency=torch.exp(torch.arange(0,w,2,device=ids.device).float()*(-math.log(10000)/w))
        angle=positions[:,None]*frequency[None,:]
        pe=torch.stack((angle.sin(),angle.cos()),dim=-1).flatten(1)[:,:w]
        return self.drop(self.token(ids)+pe.to(self.token.weight.dtype))

    def encode(self,src):
        valid=src.ne(PAD); x=self.embed(src)
        for block in self.encoder:
            if self.training and self.gradient_checkpointing:
                x=checkpoint(block,x,valid,use_reentrant=False)
            else: x=block(x,valid)
        return self.enc_norm(x),valid

    def decode(self,ids,memory,source_valid,past=None,cache=False):
        offset=0 if past is None else past[0][0][0].shape[2]
        x=self.embed(ids,offset)
        valid=ids.ne(PAD) if past is None else None
        caches=[]
        for i,block in enumerate(self.decoder):
            if self.training and self.gradient_checkpointing:
                x=checkpoint(lambda a,b,c,d,layer=block: layer(a,b,c,d)[0],
                    x,memory,source_valid,valid,use_reentrant=False)
            else:
                x,kv=block(x,memory,source_valid,valid,None if past is None else past[i],cache)
                if cache: caches.append(kv)
        return self.dec_norm(x),caches

    def forward(self,src,decoder_ids,labels=None):
        memory,valid=self.encode(src); h,_=self.decode(decoder_ids,memory,valid)
        if labels is None: return F.linear(h,self.token.weight)
        flat=h.reshape(-1,h.shape[-1]); labels=labels.reshape(-1)
        def loss_chunk(a,b):
            return F.cross_entropy(F.linear(a,self.token.weight).float(),b,ignore_index=-100,reduction='sum')
        loss=h.new_zeros((),dtype=torch.float32)
        for start in range(0,len(labels),128):
            a,b=flat[start:start+128],labels[start:start+128]
            loss=loss+(checkpoint(loss_chunk,a,b,use_reentrant=False) if self.training else loss_chunk(a,b))
        return loss


In [11]:
def sliding_attention(q,k,v,window,chunk,dropout=0.0):
    b,h,t,d=q.shape; blocks=math.ceil(t/chunk); padded=blocks*chunk
    qpad=F.pad(q,(0,0,0,padded-t))
    query=qpad.reshape(b,h,blocks,chunk,d).permute(0,2,1,3,4).reshape(b*blocks,h,chunk,d)
    positions=torch.arange(blocks,device=q.device)[:,None]*chunk-window+1+torch.arange(window+chunk-1,device=q.device)[None,:]
    indices=positions.clamp(0,t-1)
    keys=k[:,:,indices,:].permute(0,2,1,3,4).reshape(b*blocks,h,window+chunk-1,d)
    values=v[:,:,indices,:].permute(0,2,1,3,4).reshape(b*blocks,h,window+chunk-1,d)
    qi=torch.arange(chunk,device=q.device)[:,None]
    kj=torch.arange(window+chunk-1,device=q.device)[None,:]
    relative=(kj>=qi)&(kj<=qi+window-1)
    valid=(positions>=0)&(positions<t)
    mask=(relative[None,:,:]&valid[:,None,:])[None,:,:,:].expand(b,-1,-1,-1).reshape(b*blocks,1,chunk,window+chunk-1)
    output=F.scaled_dot_product_attention(query,keys,values,attn_mask=mask,dropout_p=dropout)
    return output.reshape(b,blocks,h,chunk,d).permute(0,2,1,3,4).reshape(b,h,padded,d)[:,:,:t]

class MixedAttention(nn.Module):
    def __init__(self,width,heads,window,query_chunk,global_layer,dropout):
        super().__init__(); assert width%heads==0
        self.heads=heads; self.dim=width//heads; self.window=window
        self.chunk=query_chunk; self.global_layer=global_layer; self.dropout=dropout
        self.qkv=nn.Linear(width,3*width); self.proj=nn.Linear(width,width)
    def forward(self,x,past=None,cache=False):
        b,t,w=x.shape
        q,k,v=self.qkv(x).view(b,t,3,self.heads,self.dim).unbind(2)
        q,k,v=(a.transpose(1,2) for a in (q,k,v))
        if past is not None:
            if t!=1: raise ValueError('Incremental decoding expects one token')
            k=torch.cat((past[0],k),dim=2); v=torch.cat((past[1],v),dim=2)
        p=self.dropout if self.training else 0.0
        if self.global_layer or past is not None:
            output=F.scaled_dot_product_attention(q,k,v,dropout_p=p,is_causal=past is None)
        else: output=sliding_attention(q,k,v,self.window,self.chunk,p)
        kv=None
        if cache:
            kv=(k,v) if self.global_layer else (k[:,:,-(self.window-1):].contiguous(),v[:,:,-(self.window-1):].contiguous())
        return self.proj(output.transpose(1,2).contiguous().view(b,t,w)),kv

class MixedBlock(nn.Module):
    def __init__(self,width,heads,window,query_chunk,global_layer,dropout):
        super().__init__(); self.n1=nn.LayerNorm(width); self.n2=nn.LayerNorm(width)
        self.attn=MixedAttention(width,heads,window,query_chunk,global_layer,dropout)
        self.ff=nn.Sequential(nn.Linear(width,4*width),nn.GELU(),nn.Linear(4*width,width))
        self.drop=nn.Dropout(dropout)
    def forward(self,x,past=None,cache=False):
        a,kv=self.attn(self.n1(x),past,cache); x=x+self.drop(a)
        return x+self.drop(self.ff(self.n2(x))),kv


class LocalGlobalLM(nn.Module):
    def __init__(self,vocab_size=32768,width=704,layers=16,heads=11,context=2048,window=256,
                 global_every=4,query_chunk=128,dropout=0.0,activation_checkpointing=True,loss_chunk=256):
        super().__init__()
        if min(context,query_chunk,global_every,loss_chunk)<1 or window<2: raise ValueError('Invalid architecture limits')
        self.config=dict(vocab_size=vocab_size,width=width,layers=layers,heads=heads,context=context,
            window=window,global_every=global_every,query_chunk=query_chunk,dropout=dropout,
            activation_checkpointing=activation_checkpointing,loss_chunk=loss_chunk)
        self.token=nn.Embedding(vocab_size,width)
        self.blocks=nn.ModuleList([MixedBlock(width,heads,window,query_chunk,(i+1)%global_every==0,dropout) for i in range(layers)])
        self.norm=nn.LayerNorm(width); self.drop=nn.Dropout(dropout)
        self.apply(self.initialize)
        for block in self.blocks:
            nn.init.normal_(block.attn.proj.weight,std=0.02/math.sqrt(2*layers))
            nn.init.normal_(block.ff[2].weight,std=0.02/math.sqrt(2*layers))
    @staticmethod
    def initialize(m):
        if isinstance(m,(nn.Linear,nn.Embedding)):
            nn.init.normal_(m.weight,std=0.02)
            if isinstance(m,nn.Linear) and m.bias is not None: nn.init.zeros_(m.bias)
    def hidden(self,ids,past=None,position_offset=0,cache=False):
        t=ids.shape[1]; w=self.config['width']
        if position_offset+t>self.config['context']: raise ValueError('Context exceeded')
        pos=torch.arange(position_offset,position_offset+t,device=ids.device).float()
        freq=torch.exp(torch.arange(0,w,2,device=ids.device).float()*(-math.log(10000)/w))
        phase=pos[:,None]*freq[None,:]
        pe=torch.stack((phase.sin(),phase.cos()),dim=-1).flatten(1)[:,:w]
        # Match sinusoidal amplitude with scaled learned token embeddings.
        x=self.drop(self.token(ids)*math.sqrt(w)+pe.to(self.token.weight.dtype))
        caches=[]
        for i,block in enumerate(self.blocks):
            if self.training and self.config['activation_checkpointing']:
                x=checkpoint(lambda z,layer=block: layer(z)[0],x,use_reentrant=False)
            else:
                x,kv=block(x,None if past is None else past[i],cache)
                if cache: caches.append(kv)
        return self.norm(x),caches
    def forward(self,ids,targets=None,past=None,position_offset=0,cache=False):
        h,caches=self.hidden(ids,past,position_offset,cache)
        if targets is None: return F.linear(h[:,-1],self.token.weight),caches
        flat=h.reshape(-1,h.shape[-1]); labels=targets.reshape(-1)
        def chunk_loss(a,b): return F.cross_entropy(F.linear(a,self.token.weight).float(),b,ignore_index=-100,reduction='sum')
        loss=h.new_zeros((),dtype=torch.float32); size=self.config['loss_chunk']
        for start in range(0,len(labels),size):
            a,b=flat[start:start+size],labels[start:start+size]
            loss=loss+(checkpoint(chunk_loss,a,b,use_reentrant=False) if self.training and self.config['activation_checkpointing'] else chunk_loss(a,b))
        return loss


In [12]:
class CausalGatedConv(nn.Module):
    def __init__(self,width,kernel_size,dilation):
        super().__init__()
        if kernel_size<2 or dilation<1: raise ValueError('Invalid causal convolution settings')
        self.history=(kernel_size-1)*dilation
        self.conv=nn.Conv1d(width,2*width,kernel_size,dilation=dilation,padding=0)
        self.proj=nn.Linear(width,width)

    def forward(self,x,past=None,cache=False):
        values=x.transpose(1,2)
        if past is None:
            padded=F.pad(values,(self.history,0))
        else:
            if past.shape!=(x.shape[0],x.shape[2],self.history):
                raise ValueError('Convolution cache shape mismatch')
            padded=torch.cat((past,values),dim=2)
        features=self.conv(padded)
        a,b=features.chunk(2,dim=1)
        output=self.proj((a*torch.sigmoid(b)).transpose(1,2))
        # clone prevents a small cache view from retaining a full prompt buffer.
        state=padded[:,:,-self.history:].clone() if cache else None
        return output,state

class ConvBlock(nn.Module):
    def __init__(self,width,kernel_size,dilation,dropout):
        super().__init__()
        self.n1=nn.LayerNorm(width); self.n2=nn.LayerNorm(width)
        self.causal=CausalGatedConv(width,kernel_size,dilation)
        self.ff=nn.Sequential(nn.Linear(width,4*width),nn.GELU(),nn.Linear(4*width,width))
        self.drop=nn.Dropout(dropout)

    def forward(self,x,past=None,cache=False):
        h,state=self.causal(self.n1(x),past,cache)
        x=x+self.drop(h)
        return x+self.drop(self.ff(self.n2(x))),state


class GatedConvLM(nn.Module):
    def __init__(self,vocab_size=32768,width=640,layers=16,kernel_size=3,dilation_cycle=10,
                 context=2048,dropout=0.0,activation_checkpointing=True,loss_chunk=256):
        super().__init__()
        if min(width,layers,dilation_cycle,context,loss_chunk)<1 or kernel_size<2:
            raise ValueError('Invalid model configuration')
        self.config=dict(vocab_size=vocab_size,width=width,layers=layers,kernel_size=kernel_size,
            dilation_cycle=dilation_cycle,context=context,dropout=dropout,
            activation_checkpointing=activation_checkpointing,loss_chunk=loss_chunk)
        self.dilations=[2**(i%dilation_cycle) for i in range(layers)]
        self.receptive_field=1+(kernel_size-1)*sum(self.dilations)
        self.token=nn.Embedding(vocab_size,width)
        self.blocks=nn.ModuleList([ConvBlock(width,kernel_size,d,dropout) for d in self.dilations])
        self.norm=nn.LayerNorm(width); self.drop=nn.Dropout(dropout)
        self.apply(self.initialize)
        for block in self.blocks:
            nn.init.normal_(block.causal.proj.weight,std=0.02/math.sqrt(2*layers))
            nn.init.normal_(block.ff[2].weight,std=0.02/math.sqrt(2*layers))

    @staticmethod
    def initialize(module):
        if isinstance(module,(nn.Linear,nn.Conv1d,nn.Embedding)):
            nn.init.normal_(module.weight,std=0.02)
            if getattr(module,'bias',None) is not None: nn.init.zeros_(module.bias)

    def hidden(self,ids,past=None,cache=False):
        if past is None and ids.shape[1]>self.config['context']:
            raise ValueError('Initial prompt/training block exceeds configured context')
        if past is not None and (len(past)!=len(self.blocks) or ids.shape[1]!=1):
            raise ValueError('Cached generation accepts one token and one state per layer')
        if self.training and (cache or past is not None):
            raise ValueError('Caches are for eval-mode generation only')
        x=self.drop(self.token(ids)); states=[]
        for i,block in enumerate(self.blocks):
            if self.training and self.config['activation_checkpointing']:
                x=checkpoint(lambda z,layer=block: layer(z)[0],x,use_reentrant=False)
            else:
                x,state=block(x,None if past is None else past[i],cache)
                if cache: states.append(state)
        return self.norm(x),states

    def forward(self,ids,targets=None,past=None,cache=False):
        h,states=self.hidden(ids,past,cache)
        if targets is None: return F.linear(h[:,-1],self.token.weight),states
        flat=h.reshape(-1,h.shape[-1]); labels=targets.reshape(-1)
        def chunk_loss(a,b):
            return F.cross_entropy(F.linear(a,self.token.weight).float(),b,ignore_index=-100,reduction='sum')
        loss=h.new_zeros((),dtype=torch.float32); size=self.config['loss_chunk']
        for start in range(0,len(labels),size):
            a,b=flat[start:start+size],labels[start:start+size]
            loss=loss+(checkpoint(chunk_loss,a,b,use_reentrant=False) if self.training and self.config['activation_checkpointing'] else chunk_loss(a,b))
        return loss


In [13]:
MODEL_CLASSES = {
    "model1": DecoderLM,
    "model2": EncoderDecoder,
    "model3": LocalGlobalLM,
    "model4": GatedConvLM,
}

BASES = {}
for name in MODEL_NAMES:
    folder = MODEL_DIRS[name]
    checkpoint_path = folder / "best.pt"
    tokenizer_path = folder / "tokenizer.json"
    state = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    config = copy.deepcopy(state["model_config"])
    detected = classify_model_config(config)
    if detected != name:
        raise ValueError(f"{folder} is {detected}, expected {name}")
    tokenizer = ScratchByteBPE.from_file(tokenizer_path)
    if config["vocab_size"] != tokenizer.get_vocab_size():
        raise ValueError(f"Vocabulary mismatch for {name}")
    expected_hash = state.get("tokenizer_sha256")
    if expected_hash and digest_file(tokenizer_path) != expected_hash:
        raise ValueError(f"Tokenizer hash mismatch for {name}")
    BASES[name] = {
        "folder": folder,
        "checkpoint": checkpoint_path,
        "tokenizer_path": tokenizer_path,
        "tokenizer": tokenizer,
        "config": config,
        "state": state["model"],
        "checkpoint_sha256": digest_file(checkpoint_path),
        "tokenizer_sha256": digest_file(tokenizer_path),
    }

base_report = {}
for name in MODEL_NAMES:
    model = MODEL_CLASSES[name](**BASES[name]["config"])
    result = model.load_state_dict(BASES[name]["state"], strict=True)
    assert not result.missing_keys and not result.unexpected_keys
    count = sum(parameter.numel() for parameter in model.parameters())
    base_report[name] = {"parameters": count, "config": BASES[name]["config"]}
    print(name, f"{count:,}", BASES[name]["config"])
    del model
    gc.collect()
write_json(ASSET_DIR / "phase1_models.json", base_report)


model1 119,816,576 {'vocab_size': 32768, 'width': 704, 'layers': 16, 'heads': 11, 'context': 2048, 'dropout': 0.0}
model2 122,356,608 {'vocab_size': 32768, 'width': 704, 'encoder_layers': 6, 'decoder_layers': 8, 'heads': 11, 'dropout': 0.0}
model3 118,374,784 {'vocab_size': 32768, 'width': 704, 'layers': 16, 'heads': 11, 'context': 2048, 'window': 256, 'global_every': 4, 'query_chunk': 128, 'dropout': 0.0, 'activation_checkpointing': False, 'loss_chunk': 256}
model4 119,399,680 {'vocab_size': 32768, 'width': 640, 'layers': 16, 'kernel_size': 3, 'dilation_cycle': 10, 'context': 2048, 'dropout': 0.0, 'activation_checkpointing': False, 'loss_chunk': 256}


## 6. Reuse or prepare token caches

Attached matching caches are read directly. They are checked against the tokenizer, retained case order, raw source file, and token-file hash. If no matching cache is available, preparation writes a resumable cache under the new output folder.


In [14]:
def object_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()

def prepare_tokens(model_name, split):
    base = BASES[model_name]
    folder_name = base['tokenizer_sha256'][:16]
    wanted = dict(version=VERSION, tokenizer=base['tokenizer_sha256'],
                  index=object_hash(CASE_INDEX[split]),
                  source=digest_file(DATA_ROOT / ('Test/test.json' if split == 'test' else f'Train/{split}.json')))
    candidates = []
    for cache_root in PREPARED_CACHES:
        candidates.append(Path(cache_root) / folder_name / split)
    destination = OUTPUT_ROOT / 'token_cache' / folder_name / split
    candidates.append(destination)
    for folder in candidates:
        if (folder / 'manifest.json').is_file():
            manifest = read_json(folder / 'manifest.json')
            if manifest['signature'] != wanted:
                if folder == destination:
                    raise ValueError(f'Writable cache differs from current data/tokenizer: {folder}')
                continue
            if digest_file(folder / 'tokens.bin') != manifest['sha256']:
                raise ValueError(f'Token cache hash mismatch: {folder}')
            rows = read_json(folder / 'index.json')
            if len(rows) != len(CASE_INDEX[split]): raise ValueError('Incomplete cache index')
            return folder
    destination.mkdir(parents=True, exist_ok=True)
    partial = destination / 'progress.json'
    state = read_json(partial) if partial.exists() else dict(signature=wanted, rows=[], tokens=0)
    if state['signature'] != wanted: raise ValueError(f'Partial cache settings changed: {destination}')
    rows = state['rows']
    kept = {r['id'] for r in CASE_INDEX[split]}
    path = DATA_ROOT / ('Test/test.json' if split == 'test' else f'Train/{split}.json')
    binary = destination / 'tokens.bin'
    if state['tokens'] and (not binary.exists() or binary.stat().st_size < state['tokens'] * 2):
        raise ValueError('Partial token cache is shorter than its progress record')
    with open(binary, 'r+b' if binary.exists() else 'w+b') as output:
        output.truncate(state['tokens'] * 2)
        output.seek(0, 2)
        selected_position = 0
        seen_ids = set()
        for record in tqdm(iter_records(path), total=None, desc=f'Tokenize {split}'):
            identifier = record['id']
            if identifier not in kept: continue
            if identifier in seen_ids: continue
            seen_ids.add(identifier)
            if selected_position < len(state['rows']):
                if rows[selected_position]['id'] != identifier: raise ValueError('Cache order changed')
                selected_position += 1
                continue
            ids = base['tokenizer'].encode(record['text']).ids
            if not ids or min(ids) < 0 or max(ids) >= min(65536, base['config']['vocab_size']):
                raise ValueError(f'Invalid token IDs for {identifier}')
            start = state['tokens']
            np.asarray(ids, dtype='<u2').tofile(output)
            state['tokens'] += len(ids)
            rows.append(dict(id=identifier, start=start, length=len(ids)))
            selected_position += 1
            if len(rows) % 100 == 0:
                output.flush()
                os.fsync(output.fileno())
                write_json(partial, state)
        output.flush()
        os.fsync(output.fileno())
    if [r['id'] for r in rows] != [r['id'] for r in CASE_INDEX[split]]:
        raise ValueError('Token cache does not match retained case order')
    write_json(destination / 'index.json', rows)
    lengths = [r['length'] for r in rows]
    write_json(destination / 'manifest.json', dict(signature=wanted, tokens=state['tokens'],
        sha256=digest_file(binary), token_percentiles=np.percentile(lengths,[50,90,95,99,100]).tolist()))
    return destination

TOKEN_FOLDERS = {}
def prepare_all_tokens():
    shared = {}
    for model_name in MODEL_NAMES:
        for split in ('train','dev','test'):
            key = (BASES[model_name]['tokenizer_sha256'], split)
            if key not in shared: shared[key] = prepare_tokens(model_name, split)
            TOKEN_FOLDERS[(model_name,split)] = shared[key]
    for key, folder in TOKEN_FOLDERS.items():
        print(key, read_json(folder / 'manifest.json')['token_percentiles'])


In [15]:
prepare_all_tokens()
print('Token caches are ready.')


('model1', 'train') [2473.0, 7822.0, 11023.199999999983, 23443.879999999997, 499429.0]
('model1', 'dev') [3883.0, 9491.0, 12833.5, 24404.600000000002, 60530.0]
('model1', 'test') [3927.0, 9626.600000000002, 13451.200000000012, 24350.87999999998, 56171.0]
('model2', 'train') [2473.0, 7822.0, 11023.199999999983, 23443.879999999997, 499429.0]
('model2', 'dev') [3883.0, 9491.0, 12833.5, 24404.600000000002, 60530.0]
('model2', 'test') [3927.0, 9626.600000000002, 13451.200000000012, 24350.87999999998, 56171.0]
('model3', 'train') [2473.0, 7822.0, 11023.199999999983, 23443.879999999997, 499429.0]
('model3', 'dev') [3883.0, 9491.0, 12833.5, 24404.600000000002, 60530.0]
('model3', 'test') [3927.0, 9626.600000000002, 13451.200000000012, 24350.87999999998, 56171.0]
('model4', 'train') [2473.0, 7822.0, 11023.199999999983, 23443.879999999997, 499429.0]
('model4', 'dev') [3883.0, 9491.0, 12833.5, 24404.600000000002, 60530.0]
('model4', 'test') [3927.0, 9626.600000000002, 13451.200000000012, 24350.87

In [16]:
def window_starts(length, cfg):
    size = cfg['chunk_tokens']
    if size < 2 or not 0 <= cfg['overlap'] < size: raise ValueError('Invalid chunk/overlap')
    last = max(0, length-size)
    if cfg['representation'] == 'tail': return [last]
    if cfg['representation'] == 'head_tail': return sorted(set([0,last]))
    if cfg['representation'] != 'hierarchical': raise ValueError('Unknown representation')
    if cfg['max_chunks'] < 2: raise ValueError('Hierarchical selection needs at least two chunks')
    starts = list(range(0,last+1,size-cfg['overlap']))
    if starts[-1] != last: starts.append(last)
    if len(starts) > cfg['max_chunks']:
        selected = np.linspace(0,len(starts)-1,cfg['max_chunks']).round().astype(int)
        starts = [starts[i] for i in selected]
    return starts

def max_case_chunks(cfg):
    return {'tail':1,'head_tail':2,'hierarchical':cfg['max_chunks']}[cfg['representation']]

class CaseDataset(Dataset):
    def __init__(self, model_name, split, cfg):
        folder = TOKEN_FOLDERS[(model_name,split)]
        self.tokens = np.memmap(folder/'tokens.bin',dtype='<u2',mode='r')
        self.rows = read_json(folder/'index.json')
        self.records = CASE_INDEX[split]
        self.cfg = cfg
        if [r['id'] for r in self.rows] != [r['id'] for r in self.records]:
            raise ValueError('Dataset/cache order mismatch')
    def __len__(self): return len(self.rows)
    def __getitem__(self,index):
        row = self.rows[index]
        starts = window_starts(row['length'],self.cfg)
        windows = [np.array(self.tokens[row['start']+s:row['start']+min(row['length'],s+self.cfg['chunk_tokens'])],dtype=np.int64) for s in starts]
        positions = [s/max(1,row['length']-1) for s in starts]
        label = self.records[index]['label']
        return windows, positions, (-1 if label is None else label), index

def collate_cases(samples):
    count = len(samples)
    chunks = max(len(s[0]) for s in samples)
    length = max(len(w) for s in samples for w in s[0])
    ids = torch.full((count,chunks,length),PAD,dtype=torch.long)
    mask = torch.zeros_like(ids,dtype=torch.bool)
    positions = torch.zeros(count,chunks,dtype=torch.float32)
    for i,(windows,places,_,_) in enumerate(samples):
        for j,(window,place) in enumerate(zip(windows,places)):
            ids[i,j,:len(window)] = torch.as_tensor(window)
            mask[i,j,:len(window)] = True
            positions[i,j] = place
    return dict(ids=ids, mask=mask, positions=positions,
                labels=torch.tensor([s[2] for s in samples]), indices=torch.tensor([s[3] for s in samples]))

def coverage_report(model_name,cfg):
    results = {}
    for split in ('train','dev','test'):
        rows = read_json(TOKEN_FOLDERS[(model_name,split)]/'index.json')
        covered,total,counts = 0,0,[]
        for row in rows:
            starts = window_starts(row['length'],cfg)
            end = 0
            for s in starts:
                stop = min(row['length'],s+cfg['chunk_tokens'])
                covered += max(0,stop-max(s,end)); end = max(end,stop)
            total += row['length']; counts.append(len(starts))
        results[split] = dict(token_coverage=covered/total,mean_chunks=float(np.mean(counts)),max_chunks=max(counts))
    print(json.dumps(results,indent=2))
    return results


## 7. Classification and adaptation

Model 2 combines an encoder mean with a one-token decoder query. Model 3 adapts the local/global Transformer using PEFT AdaLoRA. Model 4 keeps alternate pretrained CNN blocks to form a smaller student. The LoRMA definition is retained so your saved teacher can be loaded exactly.


In [17]:
def set_activation_checkpointing(backbone,enabled):
    if hasattr(backbone,'gradient_checkpointing'): backbone.gradient_checkpointing=enabled
    if 'activation_checkpointing' in backbone.config: backbone.config['activation_checkpointing']=enabled

class LoRMAPlus(nn.Module):
    def __init__(self,base,rank,alpha):
        super().__init__()
        self.base=base
        self.base.requires_grad_(False)
        self.A=nn.Parameter(base.weight.new_empty(rank,base.out_features))
        self.B=nn.Parameter(base.weight.new_zeros(base.out_features,rank))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))
        self.scale=alpha/rank
    def forward(self,x):
        z=F.linear(x,self.base.weight,None)
        z=z+self.scale*F.linear(F.linear(z,self.A),self.B)
        return z if self.base.bias is None else z+self.base.bias

def install_lorma(module,rank,alpha,prefix=''):
    names=[]
    for name,child in list(module.named_children()):
        path=f'{prefix}.{name}' if prefix else name
        if isinstance(child,nn.Linear):
            setattr(module,name,LoRMAPlus(child,rank,alpha)); names.append(path)
        else: names.extend(install_lorma(child,rank,alpha,path))
    return names

def decoder_hidden(backbone,ids):
    if ids.shape[1]>backbone.config['context']: raise ValueError('Context exceeded')
    pos=torch.arange(ids.shape[1],device=ids.device)
    x=backbone.drop(backbone.token(ids)+backbone.position(pos))
    for block in backbone.blocks:
        if backbone.training and backbone.gradient_checkpointing:
            x=checkpoint(lambda z,layer=block: layer(z)[0],x,use_reentrant=False)
        else: x=block(x)[0]
    return backbone.final_norm(x)

class ChunkEncoder(nn.Module):
    def __init__(self,model_name,cfg):
        super().__init__()
        base=BASES[model_name]
        self.model_name=model_name
        self.pool=cfg['model2_pool']
        self.config={'model_type':'custom_cjpe'}
        self.backbone=MODEL_CLASSES[model_name](**copy.deepcopy(base['config']))
        self.backbone.load_state_dict(base['state'],strict=True)
        set_activation_checkpointing(self.backbone,cfg['activation_checkpointing'])
        if cfg['chunk_tokens']>base['config'].get('context',1408):
            raise ValueError('Chunk length exceeds the configured/pretrained input limit')
        if cfg['student_keep'] is not None:
            if model_name!='model4': raise ValueError('Block reduction is implemented for Model 4')
            keep=cfg['student_keep']
            if not keep or sorted(set(keep))!=keep or max(keep)>=len(self.backbone.blocks):
                raise ValueError('Invalid retained student blocks')
            self.backbone.blocks=nn.ModuleList([self.backbone.blocks[i] for i in keep])
            self.backbone.dilations=[self.backbone.dilations[i] for i in keep]
            self.backbone.config['layers']=len(keep)
        if model_name=='model2' and self.pool=='encoder_only':
            self.backbone.decoder.requires_grad_(False)
            self.backbone.dec_norm.requires_grad_(False)
    def forward(self,ids,mask):
        if self.model_name=='model2':
            hidden,valid=self.backbone.encode(ids)
            if self.pool=='encoder_query':
                query=torch.full((len(ids),1),BOS,dtype=torch.long,device=ids.device)
                decoded,_=self.backbone.decode(query,hidden,valid)
                summary=decoded[:,-1].float()
            elif self.pool=='encoder_only':
                summary=(hidden.float()*mask.unsqueeze(-1)).sum(1)/mask.sum(1,keepdim=True).clamp_min(1)
            else: raise ValueError(self.pool)
        else:
            hidden=decoder_hidden(self.backbone,ids) if self.model_name=='model1' else self.backbone.hidden(ids)[0]
            last=mask.long().sum(1)-1
            summary=hidden[torch.arange(len(ids),device=ids.device),last].float()
        mean=(hidden.float()*mask.unsqueeze(-1)).sum(1)/mask.sum(1,keepdim=True).clamp_min(1)
        return torch.cat([mean,summary],dim=-1)

class CaseClassifier(nn.Module):
    def __init__(self,model_name,method,cfg,total_steps):
        super().__init__()
        self.model_name=model_name; self.method=method; self.cfg=copy.deepcopy(cfg)
        self.encoder=ChunkEncoder(model_name,cfg)
        self.adapted=[]
        if method=='lorma':
            self.encoder.requires_grad_(False)
            self.adapted=install_lorma(self.encoder.backbone.blocks,cfg['rank'],cfg['alpha'])
        elif method=='adalora':
            from peft import AdaLoraConfig
            from peft.tuners.adalora import AdaLoraModel
            targets=[n for n,m in self.encoder.named_modules() if isinstance(m,nn.Linear)]
            allocation_steps = total_steps
            if 'adalora_schedule_epochs' in cfg:
                allocation_steps = min(total_steps,
                    math.ceil(len(CASE_INDEX['train']) / cfg['effective_batch']) * cfg['adalora_schedule_epochs'])
            adaptation=AdaLoraConfig(target_modules=targets,init_r=cfg['initial_rank'],
                target_r=cfg['rank'],lora_alpha=cfg['alpha'],lora_dropout=0.0,
                total_step=allocation_steps,tinit=max(1,int(allocation_steps*.1)),
                tfinal=max(1,int(allocation_steps*.2)),deltaT=10,orth_reg_weight=cfg['orth_reg_weight'])
            self.encoder=AdaLoraModel(self.encoder,adaptation,'default')
            self.adapted=targets
        elif method not in ('full','kd','control'):
            raise ValueError(method)
        width=BASES[model_name]['config']['width']
        dim=cfg['head_dim']
        self.projection=nn.Sequential(nn.LayerNorm(2*width),nn.Dropout(cfg['dropout']),
                                      nn.Linear(2*width,dim),nn.GELU())
        self.position=nn.Linear(1,dim,bias=False)
        self.gru=nn.GRU(dim,dim//2,batch_first=True,bidirectional=True) if cfg['aggregator']=='bigru' else None
        if cfg['aggregator'] not in ('attention','bigru'): raise ValueError('Unknown aggregator')
        self.attention=nn.Sequential(nn.Linear(dim,dim//2),nn.Tanh(),nn.Linear(dim//2,1))
        self.classifier=nn.Sequential(nn.LayerNorm(dim),nn.Dropout(cfg['dropout']),nn.Linear(dim,2))
    def forward(self,ids,mask,positions):
        b,k,l=ids.shape
        valid=mask.any(-1)
        flat_valid=valid.flatten()
        features=self.encoder(ids.reshape(b*k,l)[flat_valid],mask.reshape(b*k,l)[flat_valid])
        features=self.projection(features)
        dense=features.new_zeros(b*k,features.shape[-1])
        dense[flat_valid]=features
        dense=dense.reshape(b,k,-1)+self.position(positions.unsqueeze(-1).to(features.dtype))
        if self.gru is not None:
            packed=pack_padded_sequence(dense,valid.sum(1).cpu(),batch_first=True,enforce_sorted=False)
            packed,_=self.gru(packed)
            dense,_=pad_packed_sequence(packed,batch_first=True,total_length=k)
        scores=self.attention(dense).squeeze(-1).float().masked_fill(~valid,float('-inf'))
        weights=scores.softmax(-1).to(dense.dtype)
        pooled=(dense*weights.unsqueeze(-1)).sum(1)
        return self.classifier(pooled).float()

def parameter_report(model):
    return dict(total=sum(p.numel() for p in model.parameters()),
                trainable=sum(p.numel() for p in model.parameters() if p.requires_grad),
                adapted_modules=len(model.adapted))

def make_model(model_name,method,cfg,total_steps,device):
    return CaseClassifier(model_name,method,cfg,total_steps).to(device)

def adalora_penalty(model):
    if model.method!='adalora': return next(model.parameters()).new_zeros(())
    penalties=[]
    for name,p in model.encoder.named_parameters():
        if 'lora_A' in name or 'lora_B' in name:
            w=p.float()
            gram=w@w.T if 'lora_A' in name else w.T@w
            penalties.append(torch.linalg.matrix_norm(gram-torch.eye(len(gram),device=w.device)))
    return torch.stack(penalties).mean()*model.cfg['orth_reg_weight']

def adapter_state(model):
    if model.method!='adalora': return None
    allocator=model.encoder.rankallocator
    return dict(buffers={n:{k:v.detach().cpu() for k,v in getattr(allocator,n).items()}
                         for n in ('ipt','exp_avg_ipt','exp_avg_unc')},
                rank_pattern=copy.deepcopy(model.encoder.peft_config['default'].rank_pattern))

def restore_adapter_state(model,state,device):
    if state is None: return
    for name,values in state['buffers'].items():
        setattr(model.encoder.rankallocator,name,{k:v.to(device) for k,v in values.items()})
    model.encoder.peft_config['default'].rank_pattern=state['rank_pattern']


In [18]:
def check_backbones():
    device='cuda' if torch.cuda.is_available() else 'cpu'
    for name in MODEL_NAMES:
        tok=BASES[name]['tokenizer']
        for text in ['Section 42: भारत ₹100.','a  b\n c','<s> literal text']:
            assert tok.decode(tok.encode(text).ids)==text
        cfg=dict(DEFAULT,chunk_tokens=64,representation='tail',dropout=0.0)
        model=make_model(name,'full',cfg,100,device)
        tokens=tok.encode('The court considered the appeal and the evidence.').ids[:48]
        batch=collate_cases([(np.asarray(tokens)[None,:], [0.0], 1, 0)])
        x={k:v.to(device) for k,v in batch.items() if k in ('ids','mask','positions')}
        model.eval()
        with torch.no_grad():
            original=model(**x)
            padded=dict(x,ids=F.pad(x['ids'],(0,7),value=PAD),mask=F.pad(x['mask'],(0,7),value=False))
            other=model(**padded)
        torch.testing.assert_close(original,other,rtol=2e-4,atol=2e-4)
        model.train(); F.cross_entropy(model(**x),torch.tensor([1],device=device)).backward()
        assert any(p.grad is not None and p.grad.abs().sum()>0 for p in model.classifier.parameters())
        assert all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None)
        print(name,'passed',parameter_report(model),flush=True)
        del model, x, original, other, padded
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()


## 8. Training helpers

Checkpoints save model, optimizer, learning-rate scheduler, AMP scaler, RNG state, data position, and AdaLoRA importance/allocation state. A changed run signature stops before overwriting an existing run.

The capacity test performs complete updates on maximum-length inputs at several microbatch sizes, with and without activation checkpointing. Failed numerical or memory candidates are rejected. Passing candidates need at least 1.5 GiB memory headroom per GPU. The fastest passing candidate is selected.

AMP uses a lower initial scale and bounded retries. The lifetime number of skipped updates is logged; it is not used as a stopping limit. Repeated failures on one batch still stop with a saved checkpoint.


In [19]:
def sync_cuda():
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()): torch.cuda.synchronize(i)

def autocast_context(device):
    return torch.autocast('cuda',dtype=torch.float16) if str(device).startswith('cuda') else contextlib.nullcontext()

def parallel_model(model):
    return nn.DataParallel(model) if torch.cuda.device_count()>1 and next(model.parameters()).is_cuda else model

def model_inputs(batch,device):
    return {k:batch[k].to(device) for k in ('ids','mask','positions')}

def make_optimizer(model,cfg):
    groups=[]
    for is_backbone in (True,False):
        for decay in (True,False):
            params=[p for n,p in model.named_parameters() if p.requires_grad
                    and n.startswith('encoder.')==is_backbone and (p.ndim>1)==decay]
            if params:
                lr=(cfg['adapter_lr'] if model.method in ('lorma','adalora') else cfg['backbone_lr']) if is_backbone else cfg['head_lr']
                groups.append(dict(params=params,lr=lr,weight_decay=cfg['weight_decay'] if decay else 0.0))
    return torch.optim.AdamW(groups)

def make_scheduler(optimizer,cfg,total):
    warmup=max(1,int(total*cfg['warmup_fraction']))
    def factor(step):
        if step<warmup: return (step+1)/warmup
        progress=min(1.0,(step-warmup)/max(1,total-warmup))
        return cfg['min_lr_ratio']+(1-cfg['min_lr_ratio'])*0.5*(1+math.cos(math.pi*progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer,factor)

def save_torch(path,payload):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.tmp')
    torch.save(payload,tmp); tmp.replace(path)

def rng_state():
    return dict(python=random.getstate(),numpy=np.random.get_state(),torch=torch.get_rng_state(),
                cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [])

def restore_rng(state):
    random.setstate(state['python']); np.random.set_state(state['numpy']); torch.set_rng_state(state['torch'])
    if torch.cuda.is_available() and state['cuda']:
        if len(state['cuda'])!=torch.cuda.device_count():
            raise ValueError('Resume on the same number of GPUs for reproducible continuation')
        torch.cuda.set_rng_state_all(state['cuda'])

def metric_report(labels,probabilities,threshold=0.5):
    labels=np.asarray(labels,dtype=np.int64)
    prediction=(np.asarray(probabilities)>=threshold).astype(np.int64)
    return dict(macro_f1=float(f1_score(labels,prediction,average='macro',labels=[0,1],zero_division=0)),
                precision=float(precision_score(labels,prediction,zero_division=0)),
                recall=float(recall_score(labels,prediction,zero_division=0)),
                confusion_matrix=confusion_matrix(labels,prediction,labels=[0,1]).tolist(),
                predicted_counts=np.bincount(prediction,minlength=2).tolist())

@torch.no_grad()
def predict_logits(model,dataset,batch_size,device,description='Evaluate'):
    model.eval(); inference=parallel_model(model)
    output=[]
    loader=DataLoader(dataset,batch_size=batch_size,shuffle=False,collate_fn=collate_cases,
                      num_workers=0,generator=torch.Generator().manual_seed(0))
    for batch in tqdm(loader,desc=description,leave=False):
        with autocast_context(device): logits=inference(**model_inputs(batch,device))
        if not torch.isfinite(logits).all(): raise FloatingPointError('Non-finite inference logits')
        output.append(logits.float().cpu())
    return torch.cat(output)

EXPERIMENTS={}
def register(name,model_name,method,**overrides):
    if not re.fullmatch(r'[a-z0-9_]+',name): raise ValueError('Use a simple unique run name')
    cfg=dict(DEFAULT,**overrides)
    if cfg['effective_batch'] < 1 or not 1 <= cfg['min_epochs'] <= cfg['epochs']:
        raise ValueError('Require 1 <= min_epochs <= epochs and a positive batch size')
    if cfg['patience'] < 1:
        raise ValueError('patience must be positive')
    EXPERIMENTS[name]=dict(name=name,model=model_name,method=method,config=cfg)
    return name

def run_folder(name):
    local = OUTPUT_ROOT / 'runs' / name
    if (local / 'config.json').is_file() and (local / 'best.pt').is_file():
        return local
    if name not in EXPERIMENTS and name in TEACHER_FOLDERS:
        return Path(TEACHER_FOLDERS[name])
    return local

def total_updates(cfg): return math.ceil(len(CASE_INDEX['train'])/cfg['effective_batch'])*cfg['epochs']

def hardware_signature(spec):
    return dict(version=VERSION,spec=spec,checkpoint=BASES[spec['model']]['checkpoint_sha256'],
                devices=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
                torch=torch.__version__,train_examples=len(CASE_INDEX['train']))


In [20]:
def capacity_check(name):
    spec=EXPERIMENTS[name]; cfg=spec['config']; folder=run_folder(name)
    if not torch.cuda.is_available(): raise RuntimeError('Enable a Kaggle GPU before the capacity check')
    path=folder/'hardware.json'; signature=hardware_signature(spec)
    if path.exists():
        saved=read_json(path)
        if saved['signature']!=signature: raise ValueError('Hardware/settings changed; use a new run name')
        print('Saved capacity selection:',saved['selected']); return saved['selected']
    results=[]; n_gpu=torch.cuda.device_count()
    for checkpointing in (True,False):
        for micro in (1,2,4,8):
            batch_size=micro*n_gpu
            if batch_size>cfg['effective_batch'] or cfg['effective_batch']%batch_size: continue
            trial=dict(cfg,micro_batch_per_gpu=micro,activation_checkpointing=checkpointing)
            model=net=optimizer=scaler=None
            print(f'Testing {micro} cases/GPU; checkpointing={checkpointing}',flush=True)
            try:
                gc.collect(); torch.cuda.empty_cache(); seed_everything(cfg['seed'])
                model=make_model(spec['model'],spec['method'],trial,total_updates(cfg),'cuda')
                net=parallel_model(model); model.train(); optimizer=make_optimizer(model,trial)
                scaler=torch.amp.GradScaler('cuda', init_scale=cfg['amp_initial_scale'])
                k=max_case_chunks(cfg); l=cfg['chunk_tokens']
                x=dict(ids=torch.randint(4,BASES[spec['model']]['config']['vocab_size'],(batch_size,k,l),device='cuda'),
                       mask=torch.ones(batch_size,k,l,dtype=torch.bool,device='cuda'),
                       positions=torch.linspace(0,1,k,device='cuda')[None,:].expand(batch_size,-1))
                y=torch.arange(batch_size,device='cuda')%2
                for i in range(n_gpu): torch.cuda.reset_peak_memory_stats(i)
                durations=[]
                accumulation=cfg['effective_batch']//batch_size
                successful = 0; retries = 0; consecutive = 0
                while successful < 4:
                    sync_cuda(); start=time.perf_counter(); optimizer.zero_grad(set_to_none=True)
                    for _ in range(accumulation):
                        with autocast_context('cuda'):
                            loss=F.cross_entropy(net(**x),y)/accumulation
                        if not torch.isfinite(loss):
                            raise FloatingPointError('Non-finite forward loss')
                        scaler.scale(loss).backward()
                    if spec['method']=='adalora': scaler.scale(adalora_penalty(model)).backward()
                    scaler.unscale_(optimizer)
                    norm=nn.utils.clip_grad_norm_(model.parameters(),cfg['clip'])
                    if not torch.isfinite(norm):
                        retries += 1
                        consecutive += 1
                        scale = scaler.get_scale()
                        scaler.update(new_scale=scale / 2)
                        optimizer.zero_grad(set_to_none=True)
                        print(f'AMP retry {retries}: scale {scale:g} -> {scale/2:g}', flush=True)
                        if consecutive >= cfg['max_consecutive_amp_skips']:
                            raise FloatingPointError('Gradients remained non-finite after 12 retries')
                        continue
                    scaler.step(optimizer); scaler.update()
                    if spec['method']=='adalora': model.encoder.update_and_allocate(successful+1)
                    sync_cuda(); durations.append(time.perf_counter()-start)
                    successful += 1
                    consecutive = 0
                memory=[]
                for i in range(n_gpu):
                    free,total=torch.cuda.mem_get_info(i)
                    reserved=torch.cuda.max_memory_reserved(i)
                    current=torch.cuda.memory_reserved(i)
                    headroom=min(free,total-reserved)
                    memory.append(dict(gpu=i,peak_reserved_gib=reserved/2**30,
                                       peak_tensor_gib=torch.cuda.max_memory_allocated(i)/2**30,
                                       headroom_gib=headroom/2**30))
                row=dict(micro_batch_per_gpu=micro,activation_checkpointing=checkpointing,
                         seconds_per_update=float(np.median(durations[1:])),memory=memory,amp_retries=retries,
                         passed=all(m['headroom_gib']>=cfg['capacity_headroom_gib'] for m in memory))
            except FloatingPointError as error:
                row=dict(micro_batch_per_gpu=micro,activation_checkpointing=checkpointing,passed=False,error=str(error))
            except torch.cuda.OutOfMemoryError:
                row=dict(micro_batch_per_gpu=micro,activation_checkpointing=checkpointing,passed=False,error='CUDA out of memory')
            finally:
                model=net=optimizer=scaler=None
                if 'loss' in locals(): del loss
                if 'x' in locals(): del x,y
                gc.collect(); torch.cuda.empty_cache()
            results.append(row); print(row,flush=True)
    passing=[r for r in results if r['passed']]
    if not passing: raise RuntimeError('No capacity candidate passed. Reduce chunk length/count in a new run.')
    selected=min(passing,key=lambda r:r['seconds_per_update'])
    write_json(path,dict(signature=signature,results=results,selected=selected))
    print('Selected:',selected)
    print('Training-only epoch estimate (minutes):',selected['seconds_per_update']*math.ceil(len(CASE_INDEX['train'])/cfg['effective_batch'])/60)
    return selected

def resolved_spec(name):
    spec=copy.deepcopy(EXPERIMENTS[name]); path=run_folder(name)/'hardware.json'
    if not path.is_file(): raise RuntimeError(f'Run capacity_check({name!r}) first')
    hardware=read_json(path)
    if hardware['signature']!=hardware_signature(spec): raise ValueError('Capacity settings mismatch')
    for key in ('micro_batch_per_gpu','activation_checkpointing'):
        spec['config'][key]=hardware['selected'][key]
    return spec

def training_signature(spec,teacher):
    return dict(version=VERSION,spec=spec,base_checkpoint=BASES[spec['model']]['checkpoint_sha256'],
                tokenizer=BASES[spec['model']]['tokenizer_sha256'],
                cases={split:object_hash(CASE_INDEX[split]) for split in ('train','dev')},
                tokens={split:digest_file(TOKEN_FOLDERS[(spec['model'],split)]/'tokens.bin') for split in ('train','dev')},
                teacher=teacher,
                libraries={p:importlib.metadata.version(p) for p in ['peft','transformers']})

def log_metric(folder,row):
    with open(folder/'metrics.jsonl','a',encoding='utf-8') as f: f.write(json.dumps(row)+'\n')
    print(row,flush=True)

def update_early_stopping(state, score, cfg, count_patience):
    improved = score > state['best_macro_f1'] + 1e-8
    if improved:
        state['best_macro_f1'] = score
        state['bad_epochs'] = 0
    elif count_patience:
        if state['epoch'] <= cfg['min_epochs']:
            state['bad_epochs'] = 0
        else:
            state['bad_epochs'] += 1
    return improved


def train_run(name, additional_steps=None, teacher_name=None):
    spec = resolved_spec(name)
    cfg = spec['config']
    folder = run_folder(name)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    teacher_reference = None
    teacher_logits = None
    if spec['method'] == 'kd':
        if teacher_name is None:
            raise ValueError('Select and cache the teacher before distillation.')
        teacher_logits, teacher_reference = cache_teacher(teacher_name)

    signature = training_signature(spec, teacher_reference)
    if (folder / 'signature.json').is_file() and read_json(folder / 'signature.json') != signature:
        raise ValueError('Run inputs or settings changed. Use the original settings to resume.')
    if additional_steps is not None and additional_steps < 1:
        raise ValueError('additional_steps must be positive')
    write_json(folder / 'signature.json', signature)
    write_json(folder / 'config.json', spec)
    seed_everything(cfg['seed'])
    train_data = CaseDataset(spec['model'], 'train', cfg)
    dev_data = CaseDataset(spec['model'], 'dev', cfg)
    total = total_updates(cfg)
    model = make_model(spec['model'], spec['method'], cfg, total, device)
    parameters = parameter_report(model)
    print('Parameters:', parameters, flush=True)
    if teacher_reference and parameters['total'] >= teacher_reference['parameters']:
        raise ValueError('The distilled student must be smaller than its teacher.')

    net = parallel_model(model)
    optimizer = make_optimizer(model, cfg)
    scheduler = make_scheduler(optimizer, cfg, total)
    scaler = torch.amp.GradScaler('cuda', enabled=device == 'cuda', init_scale=cfg['amp_initial_scale'])
    state = dict(epoch=0, position=0, step=0, best_macro_f1=-1.0,
                 bad_epochs=0, complete=False, skipped=0, consecutive_amp_skips=0,
                 epoch_loss_sum=0.0, epoch_examples=0)
    latest = folder / 'latest.pt'
    if latest.is_file():
        saved = torch.load(latest, map_location='cpu', weights_only=False)
        if saved['signature'] != signature:
            raise ValueError('Resume signature mismatch')
        model.load_state_dict(saved['model'], strict=True)
        optimizer.load_state_dict(saved['optimizer'])
        scheduler.load_state_dict(saved['scheduler'])
        scaler.load_state_dict(saved['scaler'])
        restore_adapter_state(model, saved['adapter'], device)
        state = saved['progress']
        restore_rng(saved['rng'])
        del saved
        print('Resumed:', state, flush=True)

    start_step = state['step']
    started = time.monotonic()
    batch_size = cfg['micro_batch_per_gpu'] * max(1, torch.cuda.device_count())
    class_weights = None
    if cfg['class_weighting']:
        counts = np.bincount([r['label'] for r in CASE_INDEX['train']], minlength=2)
        class_weights = torch.tensor(len(train_data) / (2 * counts), dtype=torch.float32, device=device)

    def save_latest():
        save_torch(latest, dict(
            signature=signature,
            model={k: v.detach().cpu() for k, v in model.state_dict().items()},
            optimizer=optimizer.state_dict(), scheduler=scheduler.state_dict(),
            scaler=scaler.state_dict(), adapter=adapter_state(model),
            progress=copy.deepcopy(state), rng=rng_state(),
        ))
        write_json(folder / 'progress.json', state)

    def validate(count_patience):
        logits = predict_logits(model, dev_data, batch_size, device)
        probabilities = logits.softmax(-1)[:, 1].numpy()
        metrics = metric_report([r['label'] for r in CASE_INDEX['dev']], probabilities)
        improved = update_early_stopping(state, metrics['macro_f1'], cfg, count_patience)
        row = dict(step=state['step'], epoch=state['epoch'],
                   validation_macro_f1=metrics['macro_f1'], best_macro_f1=state['best_macro_f1'],
                   early_stopping_active=state['epoch'] > cfg['min_epochs'],
                   bad_epochs=state['bad_epochs'])
        if count_patience and state['epoch_examples']:
            row['epoch_train_loss'] = state['epoch_loss_sum'] / state['epoch_examples']
        log_metric(folder, row)
        if improved:
            save_torch(folder / 'best.pt', dict(
                signature=signature,
                model={k: v.detach().cpu() for k, v in model.state_dict().items()},
                step=state['step'], epoch=state['epoch'], metrics=metrics, parameters=parameters,
            ))
            write_json(folder / 'best_dev_probabilities.json', [
                dict(id=r['id'], gold=r['label'], probability=float(p))
                for r, p in zip(CASE_INDEX['dev'], probabilities)
            ])
        model.train()

    try:
        if state['step'] == 0 and not (folder / 'best.pt').is_file():
            validate(False)
            save_latest()
        while not state['complete'] and state['epoch'] < cfg['epochs']:
            order = np.random.default_rng(cfg['seed'] + state['epoch']).permutation(len(train_data))
            model.train()
            while state['position'] < len(order):
                indices = order[state['position']:state['position'] + cfg['effective_batch']]
                optimizer.zero_grad(set_to_none=True)
                loss_sum = 0.0
                for begin in range(0, len(indices), batch_size):
                    selected = indices[begin:begin + batch_size]
                    batch = collate_cases([train_data[int(i)] for i in selected])
                    targets = batch['labels'].to(device)
                    with autocast_context(device):
                        logits = net(**model_inputs(batch, device))
                        ce = F.cross_entropy(logits, targets, weight=class_weights,
                            label_smoothing=cfg['label_smoothing'], reduction='none')
                        losses = ce
                        if teacher_logits is not None:
                            temperature = cfg['kd_temperature']
                            soft = (teacher_logits[selected].to(device) / temperature).softmax(-1)
                            kl = F.kl_div(F.log_softmax(logits.float() / temperature, dim=-1),
                                          soft, reduction='none').sum(-1) * temperature**2
                            losses = (1 - cfg['kd_alpha']) * ce + cfg['kd_alpha'] * kl
                        loss = losses.sum() / len(indices)
                    if not torch.isfinite(loss).item():
                        raise FloatingPointError('Non-finite forward loss. Inspect the run before continuing.')
                    scaler.scale(loss).backward()
                    loss_sum += float(loss.detach())
                if spec['method'] == 'adalora':
                    scaler.scale(adalora_penalty(model)).backward()
                scaler.unscale_(optimizer)
                norm = nn.utils.clip_grad_norm_(model.parameters(), cfg['clip'])
                before = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                skipped = scaler.get_scale() < before
                if not torch.isfinite(norm).item() and not skipped:
                    raise FloatingPointError('Non-finite gradients without an AMP skip.')
                if skipped:
                    state['skipped'] += 1
                    state['consecutive_amp_skips'] += 1
                    log_metric(folder, dict(event='amp_retry', step=state['step'], epoch=state['epoch'],
                        total_skips=state['skipped'], consecutive_skips=state['consecutive_amp_skips'],
                        scale_before=before, scale_after=scaler.get_scale()))
                    if state['consecutive_amp_skips'] >= cfg['max_consecutive_amp_skips']:
                        save_latest()
                        raise FloatingPointError('Repeated AMP failures on the same batch. Checkpoint saved; inspect retry logs.')
                    continue
                state['consecutive_amp_skips'] = 0
                state['step'] += 1
                if spec['method'] == 'adalora':
                    model.encoder.update_and_allocate(state['step'])
                scheduler.step()
                state['position'] += len(indices)
                state['epoch_loss_sum'] += loss_sum * len(indices)
                state['epoch_examples'] += len(indices)
                if state['step'] == start_step + 1 or state['step'] % cfg['log_every'] == 0:
                    log_metric(folder, dict(step=state['step'], epoch=state['epoch'], loss=loss_sum,
                        lr=optimizer.param_groups[0]['lr'], gradient_norm=float(norm),
                        skipped=state['skipped'], elapsed_minutes=(time.monotonic() - started) / 60))
                if state['step'] % cfg['save_every'] == 0:
                    save_latest()
                pause = ((additional_steps is not None and state['step'] - start_step >= additional_steps)
                         or time.monotonic() - started >= cfg['max_hours'] * 3600)
                if pause and state['position'] < len(order):
                    validate(False)
                    save_latest()
                    report = dict(name=name, **state, session_seconds=time.monotonic() - started,
                                  session_paused=True)
                    print(report, flush=True)
                    return report
            state['epoch'] += 1
            state['position'] = 0
            validate(True)
            state['epoch_loss_sum'] = 0.0
            state['epoch_examples'] = 0
            exhausted = state['epoch'] >= cfg['epochs']
            plateau = state['epoch'] > cfg['min_epochs'] and state['bad_epochs'] >= cfg['patience']
            state['complete'] = exhausted or plateau
            if state['complete']:
                state['stop_reason'] = 'epochs' if exhausted else 'early_stopping'
            save_latest()
            if additional_steps is not None and state['step'] - start_step >= additional_steps:
                break
            if time.monotonic() - started >= cfg['max_hours'] * 3600:
                break
        report = dict(name=name, **state, parameters=parameters,
                      session_seconds=time.monotonic() - started)
        write_json(folder / 'summary.json', report)
        print(report, flush=True)
        return report
    finally:
        optimizer.zero_grad(set_to_none=True)
        del net, model, optimizer, scheduler, scaler
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


## 9. Evaluator and output helpers

The professor evaluator is written verbatim and used on development predictions. Test archives contain exactly `prediction.json` at the ZIP root.


In [21]:
EVALUATOR_PATH = ASSET_DIR / "evaluate_phase4.py"
EVALUATOR_SOURCE = r'''
#!/usr/bin/env python3
"""Local evaluator for Phase 4.

Truth JSON: a list of objects, [{"id":"...", "gold":0|1}, ...]
Prediction JSON: [{"id":"...", "pred":0|1}, ...]
"""
import argparse,json,sys
from sklearn.metrics import f1_score,precision_score,recall_score

def read_truth(path):
    try:
        x=json.load(open(path,encoding="utf-8"))
    except json.JSONDecodeError as e:
        raise ValueError(f"Truth must be a JSON file containing a list of records: {e}")
    if not isinstance(x,list) or not x: raise ValueError('Truth must be a non-empty JSON list.')
    out={}
    for n,r in enumerate(x):
        if not isinstance(r,dict) or 'id' not in r or 'gold' not in r: raise ValueError(f'Truth item {n} must contain id and gold.')
        if not isinstance(r['id'],str) or not r['id'].strip() or isinstance(r['gold'],bool) or r['gold'] not in (0,1): raise ValueError(f'Invalid truth item {n}.')
        if r['id'] in out: raise ValueError(f'Duplicate truth ID: {r["id"]}')
        out[r['id']]=r['gold']
    return out

def read_pred(path):
    x=json.load(open(path,encoding='utf-8'))
    if not isinstance(x,list) or not x: raise ValueError('Prediction must be a non-empty JSON list.')
    out={}
    for n,r in enumerate(x):
        if not isinstance(r,dict) or 'id' not in r or 'pred' not in r: raise ValueError(f'Prediction item {n} must contain id and pred.')
        if not isinstance(r['id'],str) or isinstance(r['pred'],bool) or r['pred'] not in (0,1): raise ValueError(f'Invalid prediction item {n}.')
        if r['id'] in out: raise ValueError(f'Duplicate prediction ID: {r["id"]}')
        out[r['id']]=r['pred']
    return out

def main():
    ap=argparse.ArgumentParser(description='Evaluate Phase 4 binary predictions on your own held-out split.')
    ap.add_argument('--truth',required=True); ap.add_argument('--prediction',required=True); a=ap.parse_args()
    t=read_truth(a.truth); p=read_pred(a.prediction)
    if set(t)!=set(p): raise ValueError(f"ID mismatch: missing={len(set(t)-set(p))}, extra={len(set(p)-set(t))}")
    yt=[t[i] for i in t]; yp=[p[i] for i in t]
    print(f'Examples: {len(t)}')
    print(f'Macro-F1: {f1_score(yt,yp,average="macro",labels=[0,1],zero_division=0):.6f}')
    print(f'Precision: {precision_score(yt,yp,average="binary",pos_label=1,zero_division=0):.6f}')
    print(f'Recall: {recall_score(yt,yp,average="binary",pos_label=1,zero_division=0):.6f}')
if __name__=='__main__':
    try: main()
    except Exception as e: print(f'EVALUATION FAILED: {e}',file=sys.stderr); sys.exit(1)
'''.lstrip("\n")
EVALUATOR_PATH.write_text(EVALUATOR_SOURCE, encoding="utf-8")
print("Evaluator:", EVALUATOR_PATH)


Evaluator: /kaggle/working/phase4_remaining/assets/evaluate_phase4.py


In [22]:
def load_trained(name,device):
    folder=run_folder(name)
    spec=read_json(folder/'config.json'); saved=torch.load(folder/'best.pt',map_location='cpu',weights_only=False)
    if saved['signature']!=read_json(folder/'signature.json'): raise ValueError('Checkpoint signature mismatch')
    if saved['signature']['base_checkpoint']!=BASES[spec['model']]['checkpoint_sha256']:
        raise ValueError('Different Phase 1 weights attached')
    if saved['signature']['cases']['dev']!=object_hash(CASE_INDEX['dev']):
        raise ValueError('Development cases differ from the checkpoint run')
    model=make_model(spec['model'],spec['method'],spec['config'],total_updates(spec['config']),device)
    model.load_state_dict(saved['model'],strict=True)
    model.eval()
    return model,spec,saved

def validate_predictions(predictions,expected):
    if not isinstance(predictions,list) or len(predictions)!=len(expected): raise ValueError('Prediction count mismatch')
    ids=[]
    for row in predictions:
        if set(row)!={'id','pred'} or not isinstance(row['id'],str) or type(row['pred']) is not int or row['pred'] not in (0,1):
            raise ValueError(f'Invalid prediction: {row}')
        ids.append(row['id'])
    if len(set(ids))!=len(ids) or set(ids)!={r['id'] for r in expected}: raise ValueError('Prediction IDs mismatch')

def validate_run(name):
    folder=run_folder(name); device='cuda' if torch.cuda.is_available() else 'cpu'
    model,spec,saved=load_trained(name,device)
    expected_f1=saved['metrics']['macro_f1']
    try:
        cfg=spec['config']; data=CaseDataset(spec['model'],'dev',cfg)
        logits=predict_logits(model,data,cfg['micro_batch_per_gpu']*max(1,torch.cuda.device_count()),device)
        probabilities=logits.softmax(-1)[:,1].numpy()
    finally:
        del model,saved
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    truth=[r['label'] for r in CASE_INDEX['dev']]
    predictions=[dict(id=r['id'],pred=int(p>=0.5)) for r,p in zip(CASE_INDEX['dev'],probabilities)]
    validate_predictions(predictions,CASE_INDEX['dev'])
    out=folder/'validation'; out.mkdir(parents=True,exist_ok=True)
    write_json(out/'prediction.json',predictions)
    write_json(out/'probabilities.json',[dict(id=r['id'],gold=r['label'],probability=float(p)) for r,p in zip(CASE_INDEX['dev'],probabilities)])
    report=metric_report(truth,probabilities)
    report['saved_best_macro_f1']=expected_f1
    if abs(report['macro_f1']-expected_f1)>1e-6:
        print('Validation differs from the saved checkpoint score. Check numerical/device differences before submission.')
    report['per_class']=classification_report(truth,[r['pred'] for r in predictions],labels=[0,1],output_dict=True,zero_division=0)
    write_json(out/'metrics.json',report)
    result=subprocess.run([sys.executable,str(EVALUATOR_PATH),'--truth',str(ASSET_DIR/'dev_truth.json'),
                           '--prediction',str(out/'prediction.json')],capture_output=True,text=True)
    print(result.stdout,flush=True)
    if result.stderr: print(result.stderr)
    (out/'prof_evaluation.txt').write_text(result.stdout+'\n'+result.stderr,encoding='utf-8')
    write_json(out/'provenance.json',dict(checkpoint=digest_file(folder/'best.pt'),
               evaluator=digest_file(EVALUATOR_PATH),truth=digest_file(ASSET_DIR/'dev_truth.json'),
               predictions=digest_file(out/'prediction.json'),returncode=result.returncode))
    result.check_returncode()
    for label in ('0','1'): print(label,report['per_class'][label])
    return report

def threshold_comparison(name):
    folder=run_folder(name); path=folder/'validation/probabilities.json'
    if not path.exists(): validate_run(name)
    rows=read_json(path)
    results=[dict(threshold=t,**metric_report([r['gold'] for r in rows],[r['probability'] for r in rows],t))
             for t in [0.35,0.40,0.45,0.50,0.55,0.60,0.65]]
    best=max(results,key=lambda r:(r['macro_f1'],-abs(r['threshold']-.5)))
    write_json(folder/'validation/thresholds.json',dict(results=results,selected=best,
        checkpoint=digest_file(folder/'best.pt')))
    for r in results: print(r['threshold'],r['macro_f1'])
    return best['threshold']

def training_sample_report(name,per_class=64):
    device='cuda' if torch.cuda.is_available() else 'cpu'
    model,spec,saved=load_trained(name,device)
    selected=[]; rng=np.random.default_rng(42)
    for label in (0,1):
        indices=[i for i,r in enumerate(CASE_INDEX['train']) if r['label']==label]
        selected.extend(rng.choice(indices,size=min(per_class,len(indices)),replace=False).tolist())
    try:
        cfg=spec['config']
        subset=torch.utils.data.Subset(CaseDataset(spec['model'],'train',cfg),selected)
        logits=predict_logits(model,subset,cfg['micro_batch_per_gpu']*max(1,torch.cuda.device_count()),device,'Training sample')
    finally:
        del model,saved
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    gold=[CASE_INDEX['train'][i]['label'] for i in selected]
    report=metric_report(gold,logits.softmax(-1)[:,1].numpy())
    report['examples']=len(selected)
    report['case_ids']=[CASE_INDEX['train'][i]['id'] for i in selected]
    write_json(run_folder(name)/'validation/training_sample.json',report)
    print('Training sample Macro-F1:',report['macro_f1'], '| cases:',len(selected))
    return report

def export_test(name,threshold=0.5):
    folder=run_folder(name); progress=read_json(folder/'progress.json')
    if not progress['complete']:
        print('This run is paused. Exporting its current best development checkpoint.',flush=True)
    device='cuda' if torch.cuda.is_available() else 'cpu'
    model,spec,saved=load_trained(name,device)
    cfg=spec['config']
    if not 0<threshold<1: raise ValueError('Threshold must lie between zero and one')
    signature=dict(checkpoint=digest_file(folder/'best.pt'),test=object_hash(CASE_INDEX['test']),
                   test_tokens=digest_file(TOKEN_FOLDERS[(spec['model'],'test')]/'tokens.bin'),threshold=threshold)
    out=folder/'test'/object_hash(signature)[:12]; out.mkdir(parents=True,exist_ok=True)
    try:
        logits=predict_logits(model,CaseDataset(spec['model'],'test',cfg),
            cfg['micro_batch_per_gpu']*max(1,torch.cuda.device_count()),device,'Test cases')
    finally:
        del model,saved
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    probabilities=logits.softmax(-1)[:,1].numpy()
    predictions=[dict(id=r['id'],pred=int(p>=threshold)) for r,p in zip(CASE_INDEX['test'],probabilities)]
    validate_predictions(predictions,CASE_INDEX['test'])
    write_json(out/'prediction.json',predictions)
    write_json(out/'probabilities.json',[dict(id=r['id'],probability=float(p)) for r,p in zip(CASE_INDEX['test'],probabilities)])
    write_json(out/'generation.json',signature)
    archive=out/f'{name}_submission.zip'
    with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
        z.write(out/'prediction.json','prediction.json')
    with zipfile.ZipFile(archive) as z:
        assert z.namelist()==['prediction.json']
        validate_predictions(json.loads(z.read('prediction.json')),CASE_INDEX['test'])
    payload=base64.b64encode(archive.read_bytes()).decode()
    display(HTML(f'<a download="{html.escape(archive.name)}" href="data:application/zip;base64,{payload}">Download {html.escape(archive.name)}</a>'))
    print(archive)
    return archive

def completed_runs():
    rows=[]
    for path in (OUTPUT_ROOT/'runs').glob('*/best.pt'):
        folder=path.parent
        if not (folder/'progress.json').exists(): continue
        progress=read_json(folder/'progress.json'); spec=read_json(folder/'config.json')
        rows.append(dict(name=folder.name,model=spec['model'],method=spec['method'],
                    representation=spec['config']['representation'],best_macro_f1=progress['best_macro_f1'],complete=progress['complete']))
    rows.sort(key=lambda r:r['best_macro_f1'],reverse=True)
    return rows






def teacher_candidates():
    candidates = []
    names = set(TEACHER_FOLDERS)
    names.update(folder.name for folder in (OUTPUT_ROOT / 'runs').glob('*') if folder.is_dir())
    for name in sorted(names):
        folder = run_folder(name)
        if not all((folder / file).is_file() for file in (
            'best.pt', 'config.json', 'signature.json', 'progress.json')):
            continue
        spec = read_json(folder / 'config.json')
        if spec['method'] not in ('full', 'lorma', 'adalora'):
            continue
        progress = read_json(folder / 'progress.json')
        candidates.append(dict(name=name, model=spec['model'], method=spec['method'],
            best_macro_f1=progress['best_macro_f1'], complete=progress['complete'], folder=str(folder)))
    return sorted(candidates, key=lambda row: (-row['best_macro_f1'], row['name']))


def choose_teacher():
    rows = teacher_candidates()
    if TEACHER_RUN_NAME is not None:
        rows = [row for row in rows if row['name'] == TEACHER_RUN_NAME]
    if not rows:
        raise ValueError('Attach a trained LoRMA teacher backup, or finish one of the supervised runs first.')
    print('Teacher selected using development Macro-F1:', rows[0], flush=True)
    return rows[0]['name']


def validate_teacher(name):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model, spec, saved = load_trained(name, device)
    expected = saved['metrics']['macro_f1']
    try:
        cfg = spec['config']
        logits = predict_logits(model, CaseDataset(spec['model'], 'dev', cfg),
            cfg['micro_batch_per_gpu'] * max(1, torch.cuda.device_count()), device,
            'Check teacher on development cases')
        report = metric_report([row['label'] for row in CASE_INDEX['dev']], logits.softmax(-1)[:, 1].numpy())
    finally:
        del model, saved
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    report['saved_best_macro_f1'] = expected
    report['checkpoint'] = digest_file(run_folder(name) / 'best.pt')
    write_json(ASSET_DIR / 'teacher_checks' / f'{name}.json', report)
    print('Teacher saved F1:', expected, '| reproduced F1:', report['macro_f1'], flush=True)
    if abs(report['macro_f1'] - expected) > 1e-3:
        raise ValueError('Teacher validation differs by more than 0.001. Inspect the inputs before distillation.')
    return report


def cache_teacher(name):
    folder = run_folder(name)
    spec = read_json(folder / 'config.json')
    saved_signature = read_json(folder / 'signature.json')
    for split in ('train', 'dev'):
        if saved_signature['cases'][split] != object_hash(CASE_INDEX[split]):
            raise ValueError(f'Teacher {split} cases differ from the current corpus.')
    reference = dict(name=name, checkpoint=digest_file(folder / 'best.pt'),
        cases=object_hash(CASE_INDEX['train']),
        tokens=digest_file(TOKEN_FOLDERS[(spec['model'], 'train')] / 'tokens.bin'))
    cache = OUTPUT_ROOT / 'teacher_cache' / object_hash(reference)[:16]
    cache.mkdir(parents=True, exist_ok=True)
    cached = cache / 'logits.pt'
    if cached.is_file():
        saved = torch.load(cached, map_location='cpu', weights_only=False)
        if (saved['reference'] != reference or saved['logits'].shape != (len(CASE_INDEX['train']), 2)
                or not torch.isfinite(saved['logits']).all().item()):
            raise ValueError('Teacher cache mismatch or non-finite teacher logits.')
        return saved['logits'], dict(reference, parameters=saved['parameters'])

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model, spec, saved = load_trained(name, device)
    try:
        cfg = spec['config']
        parameters = parameter_report(model)['total']
        logits = predict_logits(model, CaseDataset(spec['model'], 'train', cfg),
            cfg['micro_batch_per_gpu'] * max(1, torch.cuda.device_count()), device,
            'Cache teacher on training cases')
        save_torch(cached, dict(reference=reference, logits=logits, parameters=parameters))
    finally:
        del model, saved
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return logits, dict(reference, parameters=parameters)


## 10. Check the model definitions

This uses disposable small inputs to check finite outputs, right-padding behavior, and training gradients. It does not modify saved checkpoints.


In [23]:
check_backbones()


model1 passed {'total': 120008707, 'trainable': 120008707, 'adapted_modules': 0}
model2 passed {'total': 122548739, 'trainable': 122548739, 'adapted_modules': 0}
model3 passed {'total': 118566915, 'trainable': 118566915, 'adapted_modules': 0}
model4 passed {'total': 119575171, 'trainable': 119575171, 'adapted_modules': 0}


## 11. Model 2 — full fine-tuning

Artifacts: `/kaggle/working/phase4_remaining/runs/model2_full_min10`.

Run these cells in order. Rerunning the training cell resumes `latest.pt`. The validation and submission cells load `best.pt`. A session pause can occur before ten epochs; resume training rather than treating that pause as a finished experiment.


In [24]:
register('model2_full_min10', 'model2', 'full')
coverage_report('model2', EXPERIMENTS['model2_full_min10']['config'])
capacity_check('model2_full_min10')


{
  "train": {
    "token_coverage": 0.23531127943793123,
    "mean_chunks": 1.0,
    "max_chunks": 1
  },
  "dev": {
    "token_coverage": 0.19823039756116398,
    "mean_chunks": 1.0,
    "max_chunks": 1
  },
  "test": {
    "token_coverage": 0.19616359335810005,
    "mean_chunks": 1.0,
    "max_chunks": 1
  }
}
Testing 1 cases/GPU; checkpointing=True
{'micro_batch_per_gpu': 1, 'activation_checkpointing': True, 'seconds_per_update': 1.0700381659999607, 'memory': [{'gpu': 0, 'peak_reserved_gib': 2.982421875, 'peak_tensor_gib': 2.785867691040039, 'headroom_gib': 11.35430908203125}, {'gpu': 1, 'peak_reserved_gib': 0.712890625, 'peak_tensor_gib': 0.5835566520690918, 'headroom_gib': 13.63751220703125}], 'amp_retries': 0, 'passed': True}
Testing 2 cases/GPU; checkpointing=True
{'micro_batch_per_gpu': 2, 'activation_checkpointing': True, 'seconds_per_update': 0.608762554000009, 'memory': [{'gpu': 0, 'peak_reserved_gib': 3.00390625, 'peak_tensor_gib': 2.8147687911987305, 'headroom_gib': 11.33

{'micro_batch_per_gpu': 4,
 'activation_checkpointing': False,
 'seconds_per_update': 0.2692727510000168,
 'memory': [{'gpu': 0,
   'peak_reserved_gib': 2.73828125,
   'peak_tensor_gib': 2.5060667991638184,
   'headroom_gib': 11.59844970703125},
  {'gpu': 1,
   'peak_reserved_gib': 1.75390625,
   'peak_tensor_gib': 1.555640697479248,
   'headroom_gib': 12.59649658203125}],
 'amp_retries': 0,
 'passed': True}

In [25]:
model2_full_min10_report = train_run('model2_full_min10')
print(model2_full_min10_report)


Parameters: {'total': 122548739, 'trainable': 122548739, 'adapted_modules': 0}


Evaluate:   0%|          | 0/314 [00:00<?, ?it/s]

{'step': 0, 'epoch': 0, 'validation_macro_f1': 0.3339522546419098, 'best_macro_f1': 0.3339522546419098, 'early_stopping_active': False, 'bad_epochs': 0}
{'step': 1, 'epoch': 0, 'loss': 1.0314339399337769, 'lr': 8.424599831508004e-09, 'gradient_norm': 19.36775016784668, 'skipped': 0, 'elapsed_minutes': 0.4759037404999996}
{'step': 20, 'epoch': 0, 'loss': 1.2140116691589355, 'lr': 8.845829823083403e-08, 'gradient_norm': 19.627153396606445, 'skipped': 0, 'elapsed_minutes': 0.5622112623333332}
{'step': 40, 'epoch': 0, 'loss': 1.0573786497116089, 'lr': 1.7270429654591408e-07, 'gradient_norm': 17.114665985107422, 'skipped': 0, 'elapsed_minutes': 0.6533343881333338}
{'step': 60, 'epoch': 0, 'loss': 1.0019129514694214, 'lr': 2.5695029486099416e-07, 'gradient_norm': 20.181533813476562, 'skipped': 0, 'elapsed_minutes': 0.7455958062999997}
{'step': 80, 'epoch': 0, 'loss': 0.7874135971069336, 'lr': 3.411962931760741e-07, 'gradient_norm': 7.473287582397461, 'skipped': 0, 'elapsed_minutes': 0.837315

Evaluate:   0%|          | 0/314 [00:00<?, ?it/s]

{'step': 3957, 'epoch': 1, 'validation_macro_f1': 0.332713260696253, 'best_macro_f1': 0.3339522546419098, 'early_stopping_active': False, 'bad_epochs': 0, 'epoch_train_loss': 0.6946398130100575}
{'step': 3960, 'epoch': 1, 'loss': 0.7298411130905151, 'lr': 1.6684919966301602e-05, 'gradient_norm': 2.178983449935913, 'skipped': 0, 'elapsed_minutes': 20.883045928033333}
{'step': 3980, 'epoch': 1, 'loss': 0.7014098763465881, 'lr': 1.676916596461668e-05, 'gradient_norm': 2.0595755577087402, 'skipped': 0, 'elapsed_minutes': 20.9766304474}
{'step': 4000, 'epoch': 1, 'loss': 0.643197774887085, 'lr': 1.685341196293176e-05, 'gradient_norm': 1.5613559484481812, 'skipped': 0, 'elapsed_minutes': 21.069208187799997}
{'step': 4020, 'epoch': 1, 'loss': 0.6756190657615662, 'lr': 1.6937657961246844e-05, 'gradient_norm': 2.571225881576538, 'skipped': 0, 'elapsed_minutes': 21.201238143383332}
{'step': 4040, 'epoch': 1, 'loss': 0.7576700448989868, 'lr': 1.7021903959561923e-05, 'gradient_norm': 3.93243646621

Evaluate:   0%|          | 0/314 [00:00<?, ?it/s]

... (30 outputs left out)

{'step': 67269, 'epoch': 17, 'validation_macro_f1': 0.5121818390277459, 'best_macro_f1': 0.5207863985858063, 'early_stopping_active': True, 'bad_epochs': 1, 'epoch_train_loss': 0.5418444814713793}
{'step': 67280, 'epoch': 17, 'loss': 0.5004598498344421, 'lr': 3.105430635202164e-06, 'gradient_norm': 55.6265983581543, 'skipped': 29, 'elapsed_minutes': 349.11247474863336}
{'step': 67300, 'epoch': 17, 'loss': 0.3973578214645386, 'lr': 3.101783450785414e-06, 'gradient_norm': 20.500944137573242, 'skipped': 29, 'elapsed_minutes': 349.20723968113333}
{'step': 67320, 'epoch': 17, 'loss': 1.0258617401123047, 'lr': 3.098141900620804e-06, 'gradient_norm': 63.44792938232422, 'skipped': 29, 'elapsed_minutes': 349.3400402841333}
{'step': 67340, 'epoch': 17, 'loss': 0.594808042049408, 'lr': 3.0945059873060622e-06, 'gradient_norm': 19.07950782775879, 'skipped': 29, 'elapsed_minutes': 349.4348503924666}
{'step': 67360, 'epoch': 17, 'loss': 0.6074965596199036, 'lr': 3.0908757134348928e-06, 'gradient_norm

Evaluate:   0%|          | 0/314 [00:00<?, ?it/s]

{'step': 71226, 'epoch': 18, 'validation_macro_f1': 0.5110044822434203, 'best_macro_f1': 0.5207863985858063, 'early_stopping_active': True, 'bad_epochs': 2, 'epoch_train_loss': 0.5281902772346175}
{'step': 71240, 'epoch': 18, 'loss': 0.4508728086948395, 'lr': 2.496228941375546e-06, 'gradient_norm': 6.365855693817139, 'skipped': 31, 'elapsed_minutes': 369.79805620701666}
{'step': 71260, 'epoch': 18, 'loss': 0.7331499457359314, 'lr': 2.4937427707828455e-06, 'gradient_norm': 14.244505882263184, 'skipped': 31, 'elapsed_minutes': 369.89314774353335}
{'step': 71280, 'epoch': 18, 'loss': 0.29063886404037476, 'lr': 2.491262668192674e-06, 'gradient_norm': 20.05001449584961, 'skipped': 31, 'elapsed_minutes': 369.9868620943167}
{'step': 71300, 'epoch': 18, 'loss': 0.3254317045211792, 'lr': 2.4887886353742308e-06, 'gradient_norm': 5.561864376068115, 'skipped': 31, 'elapsed_minutes': 370.0813440124666}
{'step': 71320, 'epoch': 18, 'loss': 0.3360038995742798, 'lr': 2.486320674092388e-06, 'gradient_n

KeyboardInterrupt: 

In [26]:
model2_full_min10_validation = validate_run('model2_full_min10')
model2_full_min10_training_sample = training_sample_report('model2_full_min10')


Evaluate:   0%|          | 0/314 [00:00<?, ?it/s]

Examples: 2511
Macro-F1: 0.520786
Precision: 0.521672
Recall: 0.535346

0 {'precision': 0.5200984413453651, 'recall': 0.5063897763578274, 'f1-score': 0.5131525698097936, 'support': 1252.0}
1 {'precision': 0.521671826625387, 'recall': 0.5353455123113582, 'f1-score': 0.5284202273618189, 'support': 1259.0}


Training sample:   0%|          | 0/16 [00:00<?, ?it/s]

Training sample Macro-F1: 0.8043890213338223 | cases: 128


In [27]:
model2_full_min10_submission = export_test('model2_full_min10')


This run is paused. Exporting its current best development checkpoint.


Test cases:   0%|          | 0/190 [00:00<?, ?it/s]

/kaggle/working/phase4_remaining/runs/model2_full_min10/test/ac443e8da645/model2_full_min10_submission.zip


## 12. Model 3 — AdaLoRA

Artifacts: `/kaggle/working/phase4_remaining/runs/model3_adalora_min10`.

AdaLoRA starts from the 810M-token Phase 1 checkpoint attached for Model 3. Its rank-allocation schedule spans the first 10 planned epochs, reaching the final allocation before early stopping becomes active. The learning-rate schedule uses the 20-epoch maximum. Both settings are stored in the run configuration and stay fixed on resume.


In [28]:
register('model3_adalora_min10', 'model3', 'adalora')
coverage_report('model3', EXPERIMENTS['model3_adalora_min10']['config'])
capacity_check('model3_adalora_min10')


{
  "train": {
    "token_coverage": 0.23531127943793123,
    "mean_chunks": 1.0,
    "max_chunks": 1
  },
  "dev": {
    "token_coverage": 0.19823039756116398,
    "mean_chunks": 1.0,
    "max_chunks": 1
  },
  "test": {
    "token_coverage": 0.19616359335810005,
    "mean_chunks": 1.0,
    "max_chunks": 1
  }
}
Testing 1 cases/GPU; checkpointing=True
{'micro_batch_per_gpu': 1, 'activation_checkpointing': True, 'seconds_per_update': 1.584741672999371, 'memory': [{'gpu': 0, 'peak_reserved_gib': 2.578125, 'peak_tensor_gib': 2.2560391426086426, 'headroom_gib': 11.75860595703125}, {'gpu': 1, 'peak_reserved_gib': 0.591796875, 'peak_tensor_gib': 0.5620298385620117, 'headroom_gib': 13.75860595703125}], 'amp_retries': 0, 'passed': True}
Testing 2 cases/GPU; checkpointing=True
{'micro_batch_per_gpu': 2, 'activation_checkpointing': True, 'seconds_per_update': 0.8918891210014408, 'memory': [{'gpu': 0, 'peak_reserved_gib': 2.58984375, 'peak_tensor_gib': 2.2569947242736816, 'headroom_gib': 11.7468

{'micro_batch_per_gpu': 4,
 'activation_checkpointing': False,
 'seconds_per_update': 0.5736383680014114,
 'memory': [{'gpu': 0,
   'peak_reserved_gib': 3.537109375,
   'peak_tensor_gib': 3.206399917602539,
   'headroom_gib': 10.79962158203125},
  {'gpu': 1,
   'peak_reserved_gib': 2.798828125,
   'peak_tensor_gib': 2.7127132415771484,
   'headroom_gib': 11.55157470703125}],
 'amp_retries': 0,
 'passed': True}

In [ ]:
model3_adalora_min10_report = train_run('model3_adalora_min10')
print(model3_adalora_min10_report)


Parameters: {'total': 120730435, 'trainable': 2355587, 'adapted_modules': 64}


Evaluate:   0%|          | 0/314 [00:00<?, ?it/s]

{'step': 0, 'epoch': 0, 'validation_macro_f1': 0.41466951404402974, 'best_macro_f1': 0.41466951404402974, 'early_stopping_active': False, 'bad_epochs': 0}
{'step': 1, 'epoch': 0, 'loss': 0.8840194940567017, 'lr': 4.212299915754002e-08, 'gradient_norm': 9.303632736206055, 'skipped': 0, 'elapsed_minutes': 1.1664417380333665}
{'step': 20, 'epoch': 0, 'loss': 0.6357842683792114, 'lr': 4.422914911541702e-07, 'gradient_norm': 7.424254417419434, 'skipped': 0, 'elapsed_minutes': 1.3487225331000445}
{'step': 40, 'epoch': 0, 'loss': 0.7333776950836182, 'lr': 8.635214827295703e-07, 'gradient_norm': 7.859869480133057, 'skipped': 0, 'elapsed_minutes': 1.5408878923499894}
{'step': 60, 'epoch': 0, 'loss': 0.5179902911186218, 'lr': 1.2847514743049707e-06, 'gradient_norm': 7.356203556060791, 'skipped': 0, 'elapsed_minutes': 1.7327679209667015}
{'step': 80, 'epoch': 0, 'loss': 0.8627917766571045, 'lr': 1.7059814658803707e-06, 'gradient_norm': 12.396687507629395, 'skipped': 0, 'elapsed_minutes': 1.925735

Evaluate:   0%|          | 0/314 [00:00<?, ?it/s]

{'step': 3957, 'epoch': 1, 'validation_macro_f1': 0.5921052631578947, 'best_macro_f1': 0.5921052631578947, 'early_stopping_active': False, 'bad_epochs': 0, 'epoch_train_loss': 0.6553147309851511}
{'step': 3960, 'epoch': 1, 'loss': 0.5313737392425537, 'lr': 8.342459983150801e-05, 'gradient_norm': 1.988430380821228, 'skipped': 0, 'elapsed_minutes': 41.15216885918332}
{'step': 3980, 'epoch': 1, 'loss': 0.746427059173584, 'lr': 8.384582982308341e-05, 'gradient_norm': 3.602202892303467, 'skipped': 0, 'elapsed_minutes': 41.343918475200006}
{'step': 4000, 'epoch': 1, 'loss': 0.5565167665481567, 'lr': 8.426705981465881e-05, 'gradient_norm': 3.0106711387634277, 'skipped': 0, 'elapsed_minutes': 41.5365953113167}
{'step': 4020, 'epoch': 1, 'loss': 0.659137487411499, 'lr': 8.468828980623421e-05, 'gradient_norm': 2.9627597332000732, 'skipped': 0, 'elapsed_minutes': 41.74564675875002}
{'step': 4040, 'epoch': 1, 'loss': 0.5858016014099121, 'lr': 8.510951979780961e-05, 'gradient_norm': 3.3662018775939

Evaluate:   0%|          | 0/314 [00:00<?, ?it/s]

{'step': 7914, 'epoch': 2, 'validation_macro_f1': 0.5712509712509712, 'best_macro_f1': 0.5921052631578947, 'early_stopping_active': False, 'bad_epochs': 0, 'epoch_train_loss': 0.5962466860103354}
{'step': 7920, 'epoch': 2, 'loss': 0.4564887583255768, 'lr': 9.959686858061998e-05, 'gradient_norm': 1.2501347064971924, 'skipped': 0, 'elapsed_minutes': 81.31481824206669}
{'step': 7940, 'epoch': 2, 'loss': 0.6708379983901978, 'lr': 9.959177665390047e-05, 'gradient_norm': 2.2015609741210938, 'skipped': 0, 'elapsed_minutes': 81.51292705865005}
{'step': 7960, 'epoch': 2, 'loss': 0.3410669267177582, 'lr': 9.958665291730194e-05, 'gradient_norm': 1.143660306930542, 'skipped': 0, 'elapsed_minutes': 81.70525747765}
{'step': 7980, 'epoch': 2, 'loss': 0.5913190245628357, 'lr': 9.958149737447942e-05, 'gradient_norm': 2.8088083267211914, 'skipped': 0, 'elapsed_minutes': 81.89912266348334}
{'step': 8000, 'epoch': 2, 'loss': 0.4059242606163025, 'lr': 9.957631002911067e-05, 'gradient_norm': 1.5853576660156

Evaluate:   0%|          | 0/314 [00:00<?, ?it/s]

{'step': 11871, 'epoch': 3, 'validation_macro_f1': 0.6119031097311134, 'best_macro_f1': 0.6119031097311134, 'early_stopping_active': False, 'bad_epochs': 0, 'epoch_train_loss': 0.564713936058171}
{'step': 11880, 'epoch': 3, 'loss': 0.456773966550827, 'lr': 9.797433809070575e-05, 'gradient_norm': 3.1337060928344727, 'skipped': 0, 'elapsed_minutes': 121.48947088358334}
{'step': 11900, 'epoch': 3, 'loss': 0.35313037037849426, 'lr': 9.796304780119221e-05, 'gradient_norm': 1.6344319581985474, 'skipped': 0, 'elapsed_minutes': 121.68316988655}
{'step': 11920, 'epoch': 3, 'loss': 0.4171545207500458, 'lr': 9.79517268636656e-05, 'gradient_norm': 0.9029700756072998, 'skipped': 0, 'elapsed_minutes': 121.89107315110002}
{'step': 11940, 'epoch': 3, 'loss': 0.3958585560321808, 'lr': 9.794037528620179e-05, 'gradient_norm': 3.0350024700164795, 'skipped': 0, 'elapsed_minutes': 122.08505794355}
{'step': 11960, 'epoch': 3, 'loss': 0.5025455355644226, 'lr': 9.792899307689853e-05, 'gradient_norm': 3.3407316

Evaluate:   0%|          | 0/314 [00:00<?, ?it/s]

{'step': 15828, 'epoch': 4, 'validation_macro_f1': 0.6250809196641457, 'best_macro_f1': 0.6250809196641457, 'early_stopping_active': False, 'bad_epochs': 0, 'epoch_train_loss': 0.5324495080518928}
{'step': 15840, 'epoch': 4, 'loss': 0.29906851053237915, 'lr': 9.515276540212634e-05, 'gradient_norm': 1.877179741859436, 'skipped': 3, 'elapsed_minutes': 161.66792828396666}
{'step': 15860, 'epoch': 4, 'loss': 0.5696503520011902, 'lr': 9.513559176417762e-05, 'gradient_norm': 2.4110727310180664, 'skipped': 3, 'elapsed_minutes': 161.86015484041667}
{'step': 15880, 'epoch': 4, 'loss': 0.41281425952911377, 'lr': 9.511838949520286e-05, 'gradient_norm': 3.2828948497772217, 'skipped': 3, 'elapsed_minutes': 162.05330412238334}
{'step': 15900, 'epoch': 4, 'loss': 0.38482940196990967, 'lr': 9.510115860747342e-05, 'gradient_norm': 1.6065187454223633, 'skipped': 3, 'elapsed_minutes': 162.24633096146664}
{'step': 15920, 'epoch': 4, 'loss': 0.21561521291732788, 'lr': 9.50838991132811e-05, 'gradient_norm':

In [ ]:
model3_adalora_min10_validation = validate_run('model3_adalora_min10')
model3_adalora_min10_training_sample = training_sample_report('model3_adalora_min10')


In [ ]:
model3_adalora_min10_submission = export_test('model3_adalora_min10')


## 13. Model 4 — distillation

The teacher is selected by development Macro-F1 from attached supervised checkpoints and newly trained supervised runs. Set `TEACHER_RUN_NAME` in settings to use a particular attached teacher. A paused teacher run can supply its saved best checkpoint. Teacher files remain in their attached location.

The next cell reproduces the selected teacher's development score and caches logits on training cases only. It can take several minutes. The CNN student retains alternate pretrained blocks and must have fewer parameters than its teacher. Its loss combines gold-label cross-entropy with temperature-scaled distillation loss.

If resuming a student, the selection cell reuses its recorded teacher name rather than selecting a new teacher automatically. Keep that checkpoint attached.


In [ ]:
student_signature = run_folder('model4_distilled_min10') / 'signature.json'
if student_signature.is_file():
    recorded_teacher = read_json(student_signature).get('teacher')
    if recorded_teacher is None:
        raise ValueError('Saved student signature has no teacher reference.')
    TEACHER_NAME = recorded_teacher['name']
    print('Resuming with the recorded teacher:', TEACHER_NAME)
else:
    TEACHER_NAME = choose_teacher()

teacher_validation = validate_teacher(TEACHER_NAME)
teacher_logits, teacher_reference = cache_teacher(TEACHER_NAME)
STUDENT_KEEP = list(range(0, BASES['model4']['config']['layers'], 2))
print('Retained CNN blocks:', STUDENT_KEEP)
print('Teacher reference:', teacher_reference)
del teacher_logits


In [ ]:
register('model4_distilled_min10', 'model4', 'kd', student_keep=STUDENT_KEEP)
coverage_report('model4', EXPERIMENTS['model4_distilled_min10']['config'])
capacity_check('model4_distilled_min10')


In [ ]:
model4_distilled_min10_report = train_run('model4_distilled_min10', teacher_name=TEACHER_NAME)
print(model4_distilled_min10_report)


In [ ]:
model4_distilled_min10_validation = validate_run('model4_distilled_min10')
model4_distilled_min10_training_sample = training_sample_report('model4_distilled_min10')


In [ ]:
model4_distilled_min10_submission = export_test('model4_distilled_min10')


## 14. Progress and comparison

This table reads the files, so it also works after setup in a new session. `complete=False` means training can be resumed. Best validation score can come from an earlier epoch. The completed Model 1 LoRMA submission and these three experiments form the planned four-model set.


In [ ]:
for name in REMAINING_RUN_NAMES:
    folder = run_folder(name)
    if not (folder / 'progress.json').is_file():
        print(name, 'not started')
        continue
    progress = read_json(folder / 'progress.json')
    print(name, progress)
write_json(OUTPUT_ROOT / 'experiment_comparison.json', completed_runs())
